# Aquaculture Model Analysis

This notebook demonstrates how to analyze a trained model from the aquaculture competition framework using actual competition data.

## 1. Import Libraries and Set Up Environment

This section imports all necessary libraries and sets up the environment for model analysis.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.proportion as smp
import os
import random
from pathlib import Path
import sys
import re
import joblib
import yaml
import pickle
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import optuna
import optuna.visualization as vis

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from aquaculture.feature_selection import FeatureSelector  # For feature selection capabilities
from aquaculture.config import AquacultureConfig
from src.inference import load_inference_pipeline
from src.plotting import (
    plot_feature_importance, plot_roc_curve, plot_precision_recall_curve,
    plot_confusion_matrix, plot_calibration_curve
)
from src.metrics import calculate_metrics, competition_score, calculate_roc_curve, calculate_precision_recall_curve
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve
import shap

# For reproducibility
np.random.seed(42)
random.seed(42)


## 2. Set Up Experiment Paths and Load Model

This section sets up the data and experiment directories, locates the most recent experiment, and loads the trained model and inference pipeline.

In [ ]:
# Set up paths
DATA_DIR = Path('../data')
EXPERIMENTS_DIR = Path('../experiments')

# Try to find the most recent experiment directory
if EXPERIMENTS_DIR.exists():
    experiment_dirs = [d for d in EXPERIMENTS_DIR.iterdir() if d.is_dir()]
    if experiment_dirs:
        # Sort by modification time (newest first)
        experiment_dirs.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_experiment = experiment_dirs[0]
        print(f"Found experiment: {latest_experiment.name}")
    else:
        print("No experiment directories found!")
        sys.exit(1)
else:
    print("Experiments directory not found!")
    sys.exit(1)

# Try to load the trainer first (which includes feature engineer and selector)
trainer_path = latest_experiment / "trainer.pkl"
trainer = None
if trainer_path.exists():
    try:
        with open(trainer_path, 'rb') as f:
            trainer = pickle.load(f)
        print("Trainer loaded successfully (includes feature engineer and selector)")
    except Exception as e:
        print(f"Warning: Could not load trainer.pkl: {e}")
        trainer = None

# Load the inference pipeline as fallback
print("Loading inference pipeline...")
try:
    pipeline = load_inference_pipeline(str(latest_experiment))
    print("✓ Inference pipeline loaded successfully")
    print(f"Model type: {type(pipeline.model).__name__}")
    if pipeline.feature_names:
        print(f"Number of features: {len(pipeline.feature_names)}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check that the experiment directory exists and contains a trained model")
    sys.exit(1)

## 3. Load and Prepare Training Data

This section loads the training data from CSV and prepares it for analysis by reshaping features and extracting the target variable.

In [ ]:
# Load training data from CSV file
print("Loading training data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Training data columns: {list(train_df.columns)}")

# Prepare data for training
print("Preparing data for training...")
# The target column is 'label' in the training data
# Feature columns are all columns except ID and label
feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
X_flat = train_df[feature_cols].values  # raw feature matrix (flattened)
# Reshape to 3D as expected by the feature engineer: (n_samples, 12, 12)
n_samples = X_flat.shape[0]
X = X_flat.reshape(n_samples, 12, 12)
# Get target variable - binary classification: 0 (no pond) or 1 (pond)
print("Extracting target variable from 'label' column...")
y = train_df['label'].values
print(f"Target variable shape: {y.shape}")
print(f"Target distribution: {np.bincount(y.astype(int)) if len(y) > 0 else 'empty'}")


## 4. Load Experiment Configuration and Feature Engineer

This section loads the experiment configuration and either retrieves the pre-fitted feature engineer from the trainer or creates and fits a new one.

In [ ]:
# Load experiment config to get feature engineering settings
config_path = latest_experiment / "config.yaml"
with open(config_path, 'r') as f:
    # Use FullLoader to allow construction of Python tuples (e.g., window_length_probs)
    config_dict = yaml.load(f, Loader=yaml.FullLoader)
# The TrainingConfig stores feature_engineering_config as a dict
feat_cfg_dict = config_dict.get('feature_engineering_config', {})
# If it's empty, we can also try to load via TrainingConfig (but it may not have the attr)
feature_engineering_config = AquacultureConfig(**feat_cfg_dict) if feat_cfg_dict else AquacultureConfig()
print(f"Loaded feature engineering config: {feature_engineering_config}")


In [ ]:
# Try to get the already‑fitted feature engineer from the trainer
trainer_path = latest_experiment / "trainer.pkl"
feature_engineer = None
if trainer_path.exists():
    try:
        with open(trainer_path, 'rb') as f:
            trainer_obj = pickle.load(f)
        # The trainer may have a feature_engineer attribute
        if hasattr(trainer_obj, 'feature_engineer') and trainer_obj.feature_engineer is not None:
            feature_engineer = trainer_obj.feature_engineer
            print("Retrieved fitted feature engineer from trainer.pkl")
        else:
            print("Trainer loaded but no feature_engineer attribute found.")
    except Exception as e:
        print(f"Failed to load trainer.pkl: {e}")
else:
    print("trainer.pkl not found.")

# If we don't have a fitted engineer, create a fresh one and fit it
if feature_engineer is None:
    feature_engineer = AquacultureFeatureEngineer(
        simulate_mask=feature_engineering_config.simulate_mask,
        random_state=feature_engineering_config.random_state,
        window_length_probs=feature_engineering_config.window_length_probs,
        start_month_distribution=feature_engineering_config.start_month_distribution,
        s2_monthly_dropout=feature_engineering_config.s2_monthly_dropout,
        include_optical=feature_engineering_config.include_optical,
        include_sar=feature_engineering_config.include_sar,
        include_cross_sensor_features=feature_engineering_config.include_cross_sensor_features,
        include_temporal_statistics=feature_engineering_config.include_temporal_statistics,
        include_normalized_optical=feature_engineering_config.include_normalized_optical,
        include_directional_vote=feature_engineering_config.include_directional_vote,
        include_conditional_features=feature_engineering_config.include_conditional_features,
        include_metadata=feature_engineering_config.include_metadata
    )
    print("Created new AquacultureFeatureEngineer instance.")
    # Fit on raw data (just to set internal shapes/feature names)
    feature_engineer.fit(X)
    print("Fitted feature engineer on raw data.")


## 5. Transform Features and Generate Predictions

This section transforms the raw data using the feature engineer and generates predictions using either the trainer or inference pipeline.

In [ ]:
# Transform raw data using the feature engineer with training=True
# This applies stochastic window selection and S2‑band dropout.
X_features = feature_engineer.transform(X, training=True)
X_features = X_features.values  # ensure numpy array
print(f"Reconstructed feature matrix shape: {X_features.shape}")
# Optional: show first few feature names
if hasattr(feature_engineer, 'feature_names_out_'):
    print(f"First 5 feature names: {list(feature_engineer.feature_names_out_)[:5]}")
elif hasattr(feature_engineer, 'get_feature_names_out'):
    try:
        names = feature_engineer.get_feature_names_out()
        print(f"First 5 feature names: {list(names)[:5]}")
    except Exception:
        pass


In [ ]:
# Make predictions using the reconstructed features
print("Generating predictions...")
if trainer is not None:
    # Use the trainer's predict methods which handle feature engineering and selection
    predictions = trainer.predict(X, training=True)
    probabilities = trainer.predict_proba(X, training=True)[:, 1]
    print("✓ Predictions generated using trainer (includes feature engineering and selection)")
else:
    # Fallback: use the pipeline directly on pre-computed features
    # Use the model directly to avoid double feature transformation
    predictions = pipeline.model.predict(X_features)
    probabilities = pipeline.model.predict_proba(X_features)[:, 1]
    # Note: predict_proba returns shape (n_samples, 2); we take column 1 for positive class
    print("✓ Predictions generated using pipeline (fallback method)")

## 6. Evaluate Model Performance

This section calculates and displays key performance metrics for the trained model including accuracy, precision, recall, F1-score, ROC AUC, PR AUC, and competition score.

In [ ]:
# Calculate metrics for single target
print(f"\n=== Target Evaluation ===")
target_pred = predictions
target_prob = probabilities
target_true = y

# Calculate various metrics
metrics = calculate_metrics(target_true, target_prob)

# Print key metrics
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")
print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
print(f"PR AUC:    {metrics['pr_auc']:.4f}")

# Calculate competition score (for single target, this is just the standard competition score)
competition_score_value = competition_score(target_true, target_prob)
print(f"\nCompetition Score: {competition_score_value:.4f}")

## 7. Generate Visualizations

This section creates and saves visualization plots including ROC curve, precision-recall curve, confusion matrix, and calibration curve to evaluate model performance.

In [ ]:
# Generate plots for single target
print(f"\nGenerating plots for target...")
target_pred = predictions
target_prob = probabilities
target_true = y

# Create a directory for plots
PLOTS_DIR = EXPERIMENTS_DIR / latest_experiment.name / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

# ROC Curve
fpr, tpr, _ = calculate_roc_curve(target_true, target_prob)
roc_auc = metrics['roc_auc']
plot_roc_curve(fpr, tpr, roc_auc,
               title=f'ROC Curve',
               save_path=PLOTS_DIR / f'roc_curve.png')

# Precision-Recall Curve
precision, recall, _ = calculate_precision_recall_curve(target_true, target_prob)
pr_auc = metrics['pr_auc']
plot_precision_recall_curve(precision, recall, pr_auc,
                            title=f'Precision-Recall Curve',
                            save_path=PLOTS_DIR / f'pr_curve.png')

# Confusion Matrix
cm = confusion_matrix(target_true, target_pred)
plot_confusion_matrix(cm,
                      title=f'Confusion Matrix',
                      save_path=PLOTS_DIR / f'confusion_matrix.png')

# Calibration Curve
prob_true, prob_pred = calibration_curve(target_true, target_prob, n_bins=10)
plot_calibration_curve(prob_true, prob_pred,
                       title=f'Calibration Curve',
                       save_path=PLOTS_DIR / f'calibration_curve.png')

print("\nAll plots generated successfully!")

## 8. SHAP Feature Importance Analysis

This section loads and visualizes SHAP values if they were computed during training, providing insights into feature importance for the model's predictions.

In [ ]:
# Load SHAP artifacts if they exist

exp_dir = latest_experiment  # from earlier
shap_path = exp_dir / "explanations" / "shap_values.npz"
config_path = exp_dir / "config.yaml"

if shap_path.exists():
    print(f"Loading SHAP values from {shap_path}")
    shap_data = np.load(shap_path)
    shap_values = shap_data["shap_values"]
    feature_names = shap_data["feature_names"].tolist()
    print(f"SHAP values shape: {shap_values.shape}")
    print(f"Number of features: {len(feature_names)}")
    
    # Load config to get sampling parameters
    with open(config_path, 'r') as f:
        config_dict = yaml.load(f, Loader=yaml.FullLoader)
    # The config is stored as a dict; we need to get the TrainingConfig fields
    # For simplicity, we extract what we need
    random_seed = config_dict.get('random_seed', 42)
    shap_sample_size = config_dict.get('shap_sample_size', 100)
    
    # CRITICAL: We must use the same feature transformation that was used during SHAP computation
    # The SHAP values were computed using the trainer's pipeline (feature engineering + selection)
    if trainer is not None:
        # Get features exactly as they were used during training (for SHAP consistency)
        # This applies both feature engineering and feature selection
        X_features_transformed = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
        n_samples = X_features_transformed.shape[0]
        print(f"Features after full transform (eng + select): {X_features_transformed.shape}")
        print(f"Expected features from SHAP: {shap_values.shape[1]}")
        
        # Verify we have the right number of features
        if X_features_transformed.shape[1] == shap_values.shape[1]:
            print("✓ Feature count matches between transformed data and SHAP values")
        else:
            print(f"⚠ Feature count mismatch: got {X_features_transformed.shape[1]}, expected {shap_values.shape[1]}")
            # Try to get feature names from the trainer if available
            if hasattr(trainer, 'feature_names') and trainer.feature_names is not None:
                print(f"Trainer feature names count: {len(trainer.feature_names)}")
                if trainer.feature_names is not None:
                    print(f"Trainer feature names count: {len(trainer.feature_names)}")
                    if len(trainer.feature_names) == shap_values.shape[1]:
                        feature_names = trainer.feature_names
                        print("✓ Using trainer feature names")
    else:
        # Fallback if no trainer available
        X_features_transformed = feature_engineer.transform(X, training=True).values
        n_samples = X_features_transformed.shape[0]
        print(f"Features after feature engineering only: {X_features_transformed.shape}")
    
    # Determine if sampling was applied during SHAP computation
    if shap_values.shape[0] == n_samples:
        # SHAP values were computed on the full dataset
        X_shap = X_features_transformed
        print("SHAP values computed on full dataset. Using all features for plotting.")
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = shap_values.shape[0]  # Actual number of SHAP values we have
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
            X_shap = X_features_transformed[indices]
            print(f"SHAP values computed on a sample of {sample_size} rows. "
                  f"Using the same sampled features for plotting.")
        else:
            # sample_size >= n_samples, effectively full dataset
            X_shap = X_features_transformed
            print("Sample size >= number of samples. Using full dataset for plotting.")
    
    # Final verification
    if shap_values.shape[1] == X_shap.shape[1]:
        print(f"✓ Final shapes match: SHAP {shap_values.shape} vs X_shap {X_shap.shape}")
        
        # Compute mean absolute SHAP values (feature importance)
        shap_importance = np.mean(np.abs(shap_values), axis=0)
        # Create DataFrame for easy sorting
        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": shap_importance
        }).sort_values("importance", ascending=False)
        print(importance_df.head(20))
        # Optionally save to CSV
        importance_df.to_csv(exp_dir / "explanations" / "shap_feature_importance_from_notebook.csv", index=False)
    else:
        print(f"✗ Final shape mismatch: SHAP {shap_values.shape} vs X_shap {X_shap.shape}")
        print("Cannot proceed with plotting due to feature dimension mismatch.")
else:
    print("SHAP values not found (shap_values.npz).")
    print("Please ensure that in the training notebook (01_train_model.ipynb) you have:")
    print("  config.compute_shap = True")
    print("and re-run the training to generate the SHAP values.")
    print("Alternatively, you can view the pre-generated SHAP importance CSV and plots in:")
    print(f"  {exp_dir}/explanations/shap_feature_importance.csv")
    print(f"  {exp_dir}/explanations/shap/")

## 9. Misclassification Analysis using SHAP Values

This section performs misclassification analysis using SHAP values to identify which features contribute most strongly to incorrect predictions in the training dataset.

In [ ]:
def identify_misclassifications(y_true, y_pred, y_prob, threshold=0.5):
    """
    Identify misclassified observations using predictions and probabilities.
    
    Parameters
    ----------
    y_true : array-like
        True binary labels (0 or 1)
    y_pred : array-like
        Predicted binary labels (0 or 1)
    y_prob : array-like
        Predicted probabilities for the positive class (class 1)
    threshold : float, default 0.5
        Classification threshold for converting probabilities to classes
        
    Returns
    -------
    misclassified_df : DataFrame
        DataFrame containing misclassified observations with columns:
        - original_index: Original index from the dataset
        - true_class: True class label (0 or 1)
        - predicted_class: Predicted class label (0 or 1)
        - predicted_probability: Predicted probability for class 1
        - error_type: 'false_positive' or 'false_negative'
    """
    
    # Ensure we're working with numpy arrays
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)
    
    # Identify misclassified observations
    misclassified_mask = (y_true != y_pred)
    
    # Create DataFrame with misclassified observations
    misclassified_df = pd.DataFrame({
        'original_index': np.where(misclassified_mask)[0],
        'true_class': y_true[misclassified_mask],
        'predicted_class': y_pred[misclassified_mask],
        'predicted_probability': y_prob[misclassified_mask]
    })
    
    # Add error type column
    misclassified_df['error_type'] = np.where(
        (misclassified_df['true_class'] == 0) & (misclassified_df['predicted_class'] == 1),
        'false_positive',
        'false_negative'
    )
    
    # Sort by confidence in wrong prediction
    # For false positives: sort by predicted_probability descending (most confident wrong)
    # For false negatives: sort by (1 - predicted_probability) descending (most confident wrong)
    misclassified_df['confidence_in_wrong'] = np.where(
        misclassified_df['error_type'] == 'false_positive',
        misclassified_df['predicted_probability'],
        1 - misclassified_df['predicted_probability']
    )
    
    misclassified_df = misclassified_df.sort_values('confidence_in_wrong', ascending=False)
    misclassified_df = misclassified_df.drop('confidence_in_wrong', axis=1)
    
    return misclassified_df.reset_index(drop=True)

In [ ]:
def calculate_misclassification_shap(trainer, X_raw, feature_names):
    """
    Calculate SHAP values for training observations using the trainer's pipeline.
    
    Parameters
    ----------
    trainer : object
        Trained trainer/pipeline object with _prepare_data method
    X_raw : array-like
        Raw feature data (before feature engineering)
    feature_names : list
        List of feature names corresponding to the features used by the model
        
    Returns
    -------
    shap_values : array
        SHAP values for each observation and feature
    X_features : array
        Transformed features used for SHAP calculation
    """
    
    # Get features exactly as they were used during training
    # This applies both feature engineering and feature selection
    X_features, _ = trainer._prepare_data(X_raw, np.zeros(X_raw.shape[0]), training=True)
    
    # Ensure we have the right number of features
    if X_features.shape[1] != len(feature_names):
        raise ValueError(
            f"Feature count mismatch: X_features has {X_features.shape[1]} features, "
            f"but feature_names has {len(feature_names)} features"
        )
    
    # Create a SHAP explainer using the trainer's model
    # We'll use the tree explainer if it's a tree-based model, otherwise kernel explainer
    try:
        # Try to use TreeExplainer first (faster for tree-based models)
        explainer = shap.TreeExplainer(trainer.model)
        shap_values = explainer.shap_values(X_features)
        # For binary classification, shap_values is a list of two arrays [class_0, class_1]
        # We want the SHAP values for class 1 (positive class)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]  # SHAP values for class 1
    except:
        # Fallback to KernelExplainer if TreeExplainer fails
        # Use a background sample for efficiency
        background_sample = shap.sample(X_features, min(100, X_features.shape[0]))
        explainer = shap.KernelExplainer(trainer.model.predict_proba, background_sample)
        shap_values = explainer.shap_values(X_features)
        # For binary classification, take the SHAP values for class 1
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
    
    return shap_values, X_features

In [ ]:
def explain_misclassification(index, shap_values, X_features, feature_names, y_true, y_pred, top_n=10, random_seed=42, shap_sample_size=100):
    """
    Explain an individual misclassification using SHAP values.
    """
    
    # Determine if SHAP values were computed on a subset
    n_samples = len(y_true)
    n_shap = shap_values.shape[0]
    if n_shap == n_samples:
        # SHAP values were computed on full dataset
        y_true_subset = y_true
        y_pred_subset = y_pred
        shap_index = index  # Direct indexing works
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = shap_values.shape[0]
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
        else:
            # sample_size >= n_samples, effectively full dataset
            indices = np.arange(n_samples)
        
        # Subset the labels and predictions to match the SHAP values
        y_true_subset = np.asarray(y_true)[indices]
        y_pred_subset = np.asarray(y_pred)[indices]
        
        # Convert full dataset index to subset index
        if index in indices:
            shap_index = int(np.where(indices == index)[0][0])
        else:
            # This observation is not in the SHAP subset
            raise ValueError(f"Observation index {index} is not in the SHAP subset of size {len(indices)}")
    
    # Get the SHAP values and feature values for this observation
    observation_shap = shap_values[shap_index]
    observation_features = X_features[shap_index]
    true_label = y_true_subset[shap_index]
    pred_label = y_pred_subset[shap_index]
    
    # Create DataFrame with feature information
    explanation_df = pd.DataFrame({
        'feature': feature_names,
        'feature_value': observation_features,
        'shap_value': observation_shap
    })
    
    # Add direction column
    explanation_df['direction'] = np.where(
        explanation_df['shap_value'] >= 0,
        'positive',
        'negative'
    )
    
    # Add role column based on whether the feature pushes toward wrong or correct class
    # For false positives (true=0, pred=1): positive SHAP pushes toward wrong class (class 1)
    # For false negatives (true=1, pred=0): negative SHAP pushes toward wrong class (class 0)
    explanation_df['role'] = np.where(
        ((true_label == 0) & (pred_label == 1) & (explanation_df['shap_value'] >= 0)) |
        ((true_label == 1) & (pred_label == 0) & (explanation_df['shap_value'] < 0)),
        "pushes toward wrong class",
        "pushes toward correct class"
    )
    
    # Sort by absolute SHAP value descending and take top_n
    explanation_df['abs_shap'] = np.abs(explanation_df['shap_value'])
    explanation_df = explanation_df.sort_values('abs_shap', ascending=False)
    explanation_df = explanation_df.head(top_n)
    explanation_df = explanation_df.drop('abs_shap', axis=1)
    
    return explanation_df.reset_index(drop=True)

In [ ]:
def summarize_error_shap(shap_values, X_features, feature_names, y_true, y_pred, error_type, random_seed=42, shap_sample_size=100):
    """
    Summarize SHAP values for a group of misclassified observations (false positives or false negatives).
    """
    # Determine if SHAP values were computed on a subset
    n_samples = len(y_true)
    n_shap = shap_values.shape[0]
    if n_shap == n_samples:
        # SHAP values were computed on full dataset
        y_true_subset = y_true
        y_pred_subset = y_pred
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = shap_values.shape[0]
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
        else:
            # sample_size >= n_samples, effectively full dataset
            indices = np.arange(n_samples)
        
        # Subset the labels and predictions to match the SHAP values
        y_true_subset = np.asarray(y_true)[indices]
        y_pred_subset = np.asarray(y_pred)[indices]

    # Identify the specified error type in the subset
    if error_type == 'false_positive':
        error_mask = (y_true_subset == 0) & (y_pred_subset == 1)
    elif error_type == 'false_negative':
        error_mask = (y_true_subset == 1) & (y_pred_subset == 0)
    else:
        raise ValueError("error_type must be 'false_positive' or 'false_negative'")

    # Get SHAP values and features for the error group
    error_shap = shap_values[error_mask]
    error_features = X_features[error_mask]

    # Calculate statistics for each feature
    n_error = error_shap.shape[0]

    summary_data = []
    for i, feature_name in enumerate(feature_names):
        feature_shap = error_shap[:, i]

        # Calculate basic statistics
        mean_shap = np.mean(feature_shap)
        mean_abs_shap = np.mean(np.abs(feature_shap))
        median_shap = np.median(feature_shap)

        # Calculate fraction pushing toward wrong vs correct class
        if error_type == 'false_positive':
            # For false positives: positive SHAP pushes toward wrong class (class 1)
            fraction_pushing_wrong = np.mean(feature_shap >= 0)
            fraction_pushing_correct = np.mean(feature_shap < 0)
        else:  # false_negative
            # For false negatives: negative SHAP pushes toward wrong class (class 0)
            fraction_pushing_wrong = np.mean(feature_shap < 0)
            fraction_pushing_correct = np.mean(feature_shap >= 0)

        summary_data.append({
            'feature': feature_name,
            'mean_shap': mean_shap,
            'mean_abs_shap': mean_abs_shap,
            'median_shap': median_shap,
            'fraction_pushing_wrong': fraction_pushing_wrong,
            'fraction_pushing_correct': fraction_pushing_correct
        })

    # Create DataFrame and sort by mean absolute SHAP
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values('mean_abs_shap', ascending=False)

    return summary_df.reset_index(drop=True)


In [ ]:
# Test the fixed functions with a small example
print("Testing fixed functions...")

# Create mock data to test the subsetting logic
n_full = 100
n_shap = 80  # SHAP computed on subset

# Mock data
y_true_full = np.random.randint(0, 2, n_full)
y_pred_full = np.random.randint(0, 2, n_full)
shap_values_mock = np.random.randn(n_shap, 5)  # 80 samples, 5 features
X_features_mock = np.random.randn(n_shap, 5)
feature_names_mock = [f'feature_{i}' for i in range(5)]

print(f"Full dataset size: {n_full}")
print(f"SHAP values size: {n_shap}")

# Test summarize_error_shap
try:
    fp_summary = summarize_error_shap(
        shap_values_mock, X_features_mock, feature_names_mock,
        y_true_full, y_pred_full, error_type='false_positive',
        random_seed=42, shap_sample_size=80
    )
    print("✓ summarize_error_shap works with subset")
    print(f"  Shape: {fp_summary.shape}")
except Exception as e:
    print(f"✗ summarize_error_shap failed: {e}")

# Test explain_misclassification
try:
    # Test with an index that should be in the subset
    explanation = explain_misclassification(
        10, shap_values_mock, X_features_mock, feature_names_mock,
        y_true_full, y_pred_full, top_n=3, random_seed=42, shap_sample_size=80
    )
    print("✓ explain_misclassification works with valid index")
    print(f"  Shape: {explanation.shape}")
except Exception as e:
    print(f"✗ explain_misclassification failed: {e}")

# Test analyze_feature_importance_differences
try:
    importance_diff = analyze_feature_importance_differences(
        shap_values_mock, X_features_mock, feature_names_mock,
        y_true_full, y_pred_full, random_seed=42, shap_sample_size=80
    )
    print("✓ analyze_feature_importance_differences works with subset")
    print(f"  Shape: {importance_diff.shape}")
except Exception as e:
    print(f"✗ analyze_feature_importance_differences failed: {e}")

# Test analyze_feature_value_distributions
try:
    value_dist = analyze_feature_value_distributions(
        shap_values_mock, X_features_mock, feature_names_mock,
        y_true_full, y_pred_full, random_seed=42, shap_sample_size=80
    )
    print("✓ analyze_feature_value_distributions works with subset")
    print(f"  Shape: {value_dist.shape}")
except Exception as e:
    print(f"✗ analyze_feature_value_distributions failed: {e}")

print("Testing complete!")

In [ ]:
def analyze_feature_importance_differences(shap_values, X_features, feature_names, y_true, y_pred, random_seed=42, shap_sample_size=100):
    """
    Analyze differences in feature importance between correct and incorrect predictions.

    For each feature, computes the difference in mean SHAP values between incorrect
    and correct predictions, and ranks features by the absolute difference.

    Parameters
    ----------
    shap_values : array
        SHAP values for observations (may be subset)
    X_features : array
        Feature values for observations (may be subset)
    feature_names : list
        List of feature names
    y_true : array-like
        True binary labels for full dataset
    y_pred : array-like
        Predicted binary labels for full dataset
    random_seed : int, default 42
        Random seed used for SHAP sampling
    shap_sample_size : int, default 100
        Sample size used for SHAP computation (kept for compatibility but not used)

    Returns
    -------
    importance_diff_df : DataFrame
        DataFrame containing feature importance differences with columns:
        - feature: Feature name
        - mean_shap_correct: Mean SHAP value for correct predictions
        - mean_shap_incorrect: Mean SHAP value for incorrect predictions
        - mean_shap_diff: Difference in mean SHAP (incorrect - correct)
        - abs_mean_shap_diff: Absolute difference in mean SHAP
        - mean_abs_shap_correct: Mean absolute SHAP for correct predictions
        - mean_abs_shap_incorrect: Mean absolute SHAP for incorrect predictions
    """

    # Determine if SHAP values were computed on a subset
    n_samples = len(y_true)
    n_shap = shap_values.shape[0]
    if n_shap == n_samples:
        # SHAP values were computed on full dataset
        y_true_subset = y_true
        y_pred_subset = y_pred
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = shap_values.shape[0]
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
        else:
            # sample_size >= n_samples, effectively full dataset
            indices = np.arange(n_samples)
        
        # Subset the labels and predictions to match the SHAP values
        y_true_subset = np.asarray(y_true)[indices]
        y_pred_subset = np.asarray(y_pred)[indices]

    # Identify correct and incorrect predictions
    correct_mask = (y_true_subset == y_pred_subset)
    incorrect_mask = (y_true_subset != y_pred_subset)

    # Calculate statistics for each feature
    n_correct = correct_mask.sum()
    n_incorrect = incorrect_mask.sum()

    analysis_data = []
    for i, feature_name in enumerate(feature_names):
        feature_shap = shap_values[:, i]

        # Get SHAP values for correct and incorrect predictions
        correct_shap = feature_shap[correct_mask]
        incorrect_shap = feature_shap[incorrect_mask]

        # Calculate mean SHAP values
        mean_shap_correct = np.mean(correct_shap) if n_correct > 0 else 0
        mean_shap_incorrect = np.mean(incorrect_shap) if n_incorrect > 0 else 0
        mean_shap_diff = mean_shap_incorrect - mean_shap_correct
        abs_mean_shap_diff = np.abs(mean_shap_diff)

        # Calculate mean absolute SHAP values
        mean_abs_shap_correct = np.mean(np.abs(correct_shap)) if n_correct > 0 else 0
        mean_abs_shap_incorrect = np.mean(np.abs(incorrect_shap)) if n_incorrect > 0 else 0

        analysis_data.append({
            'feature': feature_name,
            'mean_shap_correct': mean_shap_correct,
            'mean_shap_incorrect': mean_shap_incorrect,
            'mean_shap_diff': mean_shap_diff,
            'abs_mean_shap_diff': abs_mean_shap_diff,
            'mean_abs_shap_correct': mean_abs_shap_correct,
            'mean_abs_shap_incorrect': mean_abs_shap_incorrect
        })

    # Create DataFrame and sort by absolute mean SHAP difference
    importance_diff_df = pd.DataFrame(analysis_data)
    importance_diff_df = importance_diff_df.sort_values('abs_mean_shap_diff', ascending=False)

    return importance_diff_df.reset_index(drop=True)


def analyze_feature_value_distributions(shap_values, X_features, feature_names, y_true, y_pred, random_seed=42, shap_sample_size=100):
    """
    Analyze differences in feature value distributions between correct and incorrect predictions.

    For each feature, compares the distribution of feature values between correct
    and incorrect predictions to identify features with significant differences.

    Parameters
    ----------
    shap_values : array
        SHAP values for observations (may be subset)
    X_features : array
        Feature values for observations (may be subset)
    feature_names : list
        List of feature names
    y_true : array-like
        True binary labels for full dataset
    y_pred : array-like
        Predicted binary labels for full dataset
    random_seed : int, default 42
        Random seed used for SHAP sampling
    shap_sample_size : int, default 100
        Sample size used for SHAP computation (kept for compatibility but not used)

    Returns
    -------
    value_dist_df : DataFrame
        DataFrame containing feature value distribution comparisons with columns:
        - feature: Feature name
        - mean_value_correct: Mean feature value for correct predictions
        - mean_value_incorrect: Mean feature value for incorrect predictions
        - median_value_correct: Median feature value for correct predictions
        - median_value_incorrect: Median feature value for incorrect predictions
        - std_value_correct: Standard deviation of feature values for correct predictions
        - std_value_incorrect: Standard deviation of feature values for incorrect predictions
        - mean_abs_diff: Absolute difference in mean values
    """

    # Determine if SHAP values were computed on a subset
    n_samples = len(y_true)
    n_shap = shap_values.shape[0]
    if n_shap == n_samples:
        # SHAP values were computed on full dataset
        y_true_subset = y_true
        y_pred_subset = y_pred
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = shap_values.shape[0]
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
        else:
            # sample_size >= n_samples, effectively full dataset
            indices = np.arange(n_samples)
        
        # Subset the labels and predictions to match the SHAP values
        y_true_subset = np.asarray(y_true)[indices]
        y_pred_subset = np.asarray(y_pred)[indices]

    # Identify correct and incorrect predictions
    correct_mask = (y_true_subset == y_pred_subset)
    incorrect_mask = (y_true_subset != y_pred_subset)

    # Calculate statistics for each feature
    n_correct = correct_mask.sum()
    n_incorrect = incorrect_mask.sum()

    analysis_data = []
    for i, feature_name in enumerate(feature_names):
        feature_values = X_features[:, i]

        # Get feature values for correct and incorrect predictions
        correct_values = feature_values[correct_mask]
        incorrect_values = feature_values[incorrect_mask]

        # Calculate statistics for correct predictions
        mean_value_correct = np.mean(correct_values) if n_correct > 0 else 0
        median_value_correct = np.median(correct_values) if n_correct > 0 else 0
        std_value_correct = np.std(correct_values) if n_correct > 0 else 0

        # Calculate statistics for incorrect predictions
        mean_value_incorrect = np.mean(incorrect_values) if n_incorrect > 0 else 0
        median_value_incorrect = np.median(incorrect_values) if n_incorrect > 0 else 0
        std_value_incorrect = np.std(incorrect_values) if n_incorrect > 0 else 0

        # Calculate absolute difference in mean values
        mean_abs_diff = np.abs(mean_value_incorrect - mean_value_correct)

        analysis_data.append({
            'feature': feature_name,
            'mean_value_correct': mean_value_correct,
            'mean_value_incorrect': mean_value_incorrect,
            'median_value_correct': median_value_correct,
            'median_value_incorrect': median_value_incorrect,
            'std_value_correct': std_value_correct,
            'std_value_incorrect': std_value_incorrect,
            'mean_abs_diff': mean_abs_diff
        })

    # Create DataFrame and sort by absolute mean difference
    value_dist_df = pd.DataFrame(analysis_data)
    value_dist_df = value_dist_df.sort_values('mean_abs_diff', ascending=False)

    return value_dist_df.reset_index(drop=True)


In [ ]:
def analyze_directional_vote_errors(shap_values, X_features, feature_names, y_true, y_pred, random_seed=42, shap_sample_size=100):
    """
    Analyze directional-vote features specifically for different prediction outcomes.
    
    Parameters
    ----------
    shap_values : array
        SHAP values for observations (may be subset)
    X_features : array
        Feature values for observations (may be subset)
    feature_names : list
        List of feature names
    y_true : array-like
        True binary labels for full dataset
    y_pred : array-like
        Predicted binary labels for full dataset
    random_seed : int, default 42
        Random seed used for SHAP sampling
    shap_sample_size : int, default 100
        Sample size used for SHAP computation
        
    Returns
    -------
    directional_vote_df : DataFrame
        DataFrame containing analysis of directional-vote features for each outcome class:
        - feature: Feature name
        - outcome_class: True/Predicted class combination ('true_pos', 'true_neg', 'false_pos', 'false_neg')
        - mean_feature_value: Mean feature value for this group
        - median_feature_value: Median feature value for this group
        - mean_shap: Mean SHAP value for this group
        - mean_abs_shap: Mean absolute SHAP value for this group
        - fraction_shap_toward_class1: Fraction of SHAP values pushing toward class 1
        - fraction_shap_toward_class0: Fraction of SHAP values pushing toward class 0
    """
    
    # Define the six directional-vote features we're interested in
    directional_vote_features = [
        'directional_vote_fraction_ge_2',
        'directional_vote_fraction_positive',
        'directional_vote_min',
        'directional_vote_fraction_eq_4',
        'directional_vote_max',
        'directional_vote_mean'
    ]
    
    # Filter to only include directional-vote features that exist in our feature set
    directional_vote_features = [f for f in directional_vote_features if f in feature_names]
    
    if not directional_vote_features:
        raise ValueError("No directional-vote features found in the provided feature names")
    
    # Determine if SHAP values were computed on a subset
    n_samples = len(y_true)
    if shap_values.shape[0] == n_samples:
        # SHAP values were computed on full dataset
        y_true_subset = y_true
        y_pred_subset = y_pred
        indices = np.arange(n_samples)  # All indices
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = min(shap_sample_size, n_samples)
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
        else:
            # sample_size >= n_samples, effectively full dataset
            indices = np.arange(n_samples)
        
        # Subset the labels and predictions to match the SHAP values
        y_true_subset = np.asarray(y_true)[indices]
        y_pred_subset = np.asarray(y_pred)[indices]
    
    # Get indices of directional-vote features
    directional_vote_indices = [feature_names.index(f) for f in directional_vote_features]
    
    # Define outcome classes
    outcome_conditions = {
        'true_pos': (y_true_subset == 1) & (y_pred_subset == 1),
        'true_neg': (y_true_subset == 0) & (y_pred_subset == 0),
        'false_pos': (y_true_subset == 0) & (y_pred_subset == 1),
        'false_neg': (y_true_subset == 1) & (y_pred_subset == 0)
    }
    
    # Analyze each directional-vote feature for each outcome class
    analysis_data = []
    
    for feature_name in directional_vote_features:
        feature_idx = feature_names.index(feature_name)
        feature_shap = shap_values[:, feature_idx]
        feature_values = X_features[:, feature_idx]
        
        for outcome_class, condition in outcome_conditions.items():
            # Get values for this outcome class
            outcome_shap = feature_shap[condition]
            outcome_values = feature_values[condition]
            
            if len(outcome_shap) == 0:
                # No observations in this group
                mean_feature_value = np.nan
                median_feature_value = np.nan
                mean_shap = np.nan
                mean_abs_shap = np.nan
                fraction_shap_toward_class1 = np.nan
                fraction_shap_toward_class0 = np.nan
            else:
                # Calculate statistics
                mean_feature_value = np.mean(outcome_values)
                median_feature_value = np.median(outcome_values)
                mean_shap = np.mean(outcome_shap)
                mean_abs_shap = np.mean(np.abs(outcome_shap))
                
                # Calculate fraction of SHAP values pushing toward each class
                fraction_shap_toward_class1 = np.mean(outcome_shap >= 0)
                fraction_shap_toward_class0 = np.mean(outcome_shap < 0)
            
            analysis_data.append({
                'feature': feature_name,
                'outcome_class': outcome_class,
                'mean_feature_value': mean_feature_value,
                'median_feature_value': median_feature_value,
                'mean_shap': mean_shap,
                'mean_abs_shap': mean_abs_shap,
                'fraction_shap_toward_class1': fraction_shap_toward_class1,
                'fraction_shap_toward_class0': fraction_shap_toward_class0
            })
    
    # Create DataFrame
    directional_vote_df = pd.DataFrame(analysis_data)
    
    return directional_vote_df.reset_index(drop=True)

In [ ]:
def plot_misclassification_shap(shap_values, X_features, feature_names, y_true, y_pred, observation_index, save_path=None, random_seed=42, shap_sample_size=100):
    """
    Create a SHAP-style horizontal bar plot for a specific misclassified observation.
    
    Parameters
    ----------
    shap_values : array
        SHAP values for observations (may be subset)
    X_features : array
        Feature values for observations (may be subset)
    feature_names : list
        List of feature names
    y_true : array-like
        True binary labels for full dataset
    y_pred : array-like
        Predicted binary labels for full dataset
    observation_index : int
        Index of the observation to plot (in full dataset)
    save_path : str or Path, optional
        Path to save the plot. If None, the plot is not saved.
    random_seed : int, default 42
        Random seed used for SHAP sampling
    shap_sample_size : int, default 100
        Sample size used for SHAP computation
        
    Returns
    -------
    fig : matplotlib.figure.Figure
        The generated figure
    """
    
    # Get the explanation for this observation
    explanation_df = explain_misclassification(
        observation_index, shap_values, X_features, feature_names, y_true, y_pred, top_n=15, random_seed=random_seed, shap_sample_size=shap_sample_size
    )
    
    # Get the true and predicted labels for this observation
    # Note: We need to determine the true label for the specific observation
    # Since observation_index is now a subset index, we need to get the corresponding label
    # But explain_misclassification already handled the subsetting internally and returned the explanation
    # For the plot title, we need the actual true/pred labels for this observation
    
    # Determine if SHAP values were computed on a subset to get the correct labels
    n_samples = len(y_true)
    if shap_values.shape[0] == n_samples:
        # SHAP values were computed on full dataset
        true_label = y_true[observation_index]
        pred_label = y_pred[observation_index]
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = min(shap_sample_size, n_samples)
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
        else:
            # sample_size >= n_samples, effectively full dataset
            indices = np.arange(n_samples)
        
        # Subset the labels and predictions to match the SHAP values
        y_true_subset = np.asarray(y_true)[indices]
        y_pred_subset = np.asarray(y_pred)[indices]
        
        # Convert full dataset index to subset index
        if observation_index in indices:
            shap_index = int(np.where(indices == observation_index)[0][0])
        else:
            # This observation is not in the SHAP subset
            raise ValueError(f"Observation index {observation_index} is not in the SHAP subset of size {len(indices)}")
        
        # Get the labels for this observation from the subset
        true_label = y_true_subset[shap_index]
        pred_label = y_pred_subset[shap_index]
    
    # Determine what the "wrong class" is for labeling purposes
    if true_label == 0 and pred_label == 1:
        wrong_class_label = "Class 1 (Wrong)"
        correct_class_label = "Class 0 (Correct)"
    elif true_label == 1 and pred_label == 0:
        wrong_class_label = "Class 0 (Wrong)"
        correct_class_label = "Class 1 (Correct)"
    else:
        # This shouldn't happen for misclassified observations, but handle it anyway
        wrong_class_label = f"Class {pred_label}"
        correct_class_label = f"Class {1-pred_label}"
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Colors: red for pushing toward wrong class, blue for pushing toward correct class
    colors = ['red' if role == "pushes toward wrong class" else 'blue' 
              for role in explanation_df['role']]
    
    # Create horizontal bar plot
    y_pos = np.arange(len(explanation_df))
    ax.barh(y_pos, explanation_df['shap_value'], color=colors, alpha=0.7)
    
    # Customize the plot
    ax.set_yticks(y_pos)
    ax.set_yticklabels(explanation_df['feature'])
    ax.set_xlabel('SHAP Value')
    ax.set_title(f'SHAP Values for Observation {observation_index}\n'
                 f'True Class: {true_label}, Predicted Class: {pred_label}')
    ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    
    # Add a legend
    legend_elements = [
        Patch(facecolor='red', alpha=0.7, label=f'Pushes toward {wrong_class_label}'),
        Patch(facecolor='blue', alpha=0.7, label=f'Pushes toward {correct_class_label}')
    ]
    ax.legend(handles=legend_elements, loc='lower right')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save if requested
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    return fig

In [ ]:
def save_analysis_results(results_dict, output_dir):
    """
    Save analysis results to CSV files and plots to PNG files.
    
    Parameters
    ----------
    results_dict : dict
        Dictionary containing the analysis results to save.
        Expected keys:
        - 'misclassified_observations': DataFrame to save as misclassified_observations.csv
        - 'false_positive_shap_summary': DataFrame to save as false_positive_shap_summary.csv
        - 'false_negative_shap_summary': DataFrame to save as false_negative_shap_summary.csv
        - 'directional_vote_error_analysis': DataFrame to save as directional_vote_error_analysis.csv
        - 'top_false_positives': DataFrame to save as top_false_positives.csv
        - 'top_false_negatives': DataFrame to save as top_false_negatives.csv
        - Any matplotlib figures should be stored with keys ending in '_fig' 
          (e.g., 'most_confident_fp_fig', 'most_confident_fn_fig')
    output_dir : str or Path
        Directory where results should be saved
        
    Returns
    -------
    saved_files : list
        List of file paths that were saved
    """
    
    # Create output directory if it doesn't exist
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    saved_files = []
    
    # Save each DataFrame in the results dictionary
    for key, value in results_dict.items():
        if isinstance(value, pd.DataFrame):
            # Save DataFrame as CSV
            file_path = output_dir / f"{key}.csv"
            value.to_csv(file_path, index=False)
            saved_files.append(file_path)
            print(f"Saved {key} to {file_path}")
        elif hasattr(value, 'savefig') and hasattr(value, 'axes'):
            # Assume it's a matplotlib figure
            file_path = output_dir / f"{key}.png"
            value.savefig(file_path, dpi=150, bbox_inches='tight')
            saved_files.append(file_path)
            print(f"Saved {key} to {file_path}")
    
    return saved_files

## 10. Misclassification Analysis Execution

This section executes the misclassification analysis using the functions defined above to identify and visualize incorrectly classified observations.

In [ ]:
# Get the indices of the most confidently wrong observations
# Ensure misclassified_df is available
if 'misclassified_df' not in locals():
    # Compute misclassified_df if not already computed
    misclassified_df = identify_misclassifications(y, predictions, probabilities)

# Compute false positive and false negative subsets
if 'misclassified_df' in locals() and len(misclassified_df) > 0:
    fp_misclassified = misclassified_df[misclassified_df['error_type'] == 'false_positive']
    fn_misclassified = misclassified_df[misclassified_df['error_type'] == 'false_negative']
    fp_count = len(fp_misclassified)
    fn_count = len(fn_misclassified)
else:
    fp_misclassified = pd.DataFrame()
    fn_misclassified = pd.DataFrame()
    fp_count = 0
    fn_count = 0

# Initialize indices
most_confident_fp_idx = None
most_confident_fn_idx = None

# For SHAP plots, we need to find the most confident wrong observation THAT IS IN THE SHAP SUBSET
# We'll compute this after we know the SHAP subset indices (if SHAP data is available)
if 'shap_values' in locals() and 'X_shap' in locals() and 'feature_names' in locals():
    # Determine the SHAP subset indices
    n_samples = len(y)
    n_shap = shap_values.shape[0]
    if n_shap == n_samples:
        # SHAP values were computed on full dataset
        shap_indices = np.arange(n_samples)
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        # These variables should be set in the shap-load cell
        try:
            current_random_seed = random_seed
            current_shap_sample_size = shap_sample_size
        except NameError:
            # Fallback defaults
            current_random_seed = 42
            current_shap_sample_size = 100
        
        if current_shap_sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(current_random_seed)
            shap_indices = rng.choice(n_samples, size=current_shap_sample_size, replace=False)
        else:
            # current_shap_sample_size >= n_samples, effectively full dataset
            shap_indices = np.arange(n_samples)
    
    # Now find the most confident wrong observation within the SHAP subset
    if fp_count > 0:
        # Filter fp_misclassified to only include observations in SHAP subset
        fp_in_shap = fp_misclassified[fp_misclassified['original_index'].isin(shap_indices)]
        if len(fp_in_shap) > 0:
            most_confident_fp_idx = fp_in_shap.loc[
                fp_in_shap['predicted_probability'].idxmax()
            ]['original_index']
            print(f"Creating visualization for most confidently wrong false positive IN SHAP SUBSET (index {most_confident_fp_idx})")
        else:
            print("No false positives found in SHAP subset.")
    
    if fn_count > 0:
        # Filter fn_misclassified to only include observations in SHAP subset
        fn_in_shap = fn_misclassified[fn_misclassified['original_index'].isin(shap_indices)]
        if len(fn_in_shap) > 0:
            most_confident_fn_idx = fn_in_shap.loc[
                fn_in_shap['predicted_probability'].idxmin()
            ]['original_index']
            print(f"Creating visualization for most confidently wrong false negative IN SHAP SUBSET (index {most_confident_fn_idx})")
        else:
            print("No false negatives found in SHAP subset.")
else:
    # Fallback to original behavior if SHAP data not available (for summary section)
    if fp_count > 0:
        most_confident_fp_idx = fp_misclassified.loc[
            fp_misclassified['predicted_probability'].idxmax()
        ]['original_index']
        print(f"Creating visualization for most confidently wrong false positive (index {most_confident_fp_idx})")
    
    if fn_count > 0:
        # For false negatives, most confident wrong = lowest predicted probability
        # (most sure it should be class 1 but predicted 0)
        most_confident_fn_idx = fn_misclassified.loc[
            fn_misclassified['predicted_probability'].idxmin()
        ]['original_index']
        print(f"Creating visualization for most confidently wrong false negative (index {most_confident_fn_idx})")

In [ ]:
# Compute SHAP summaries and analysis for misclassified observations
# Only proceed if SHAP values are available
if 'shap_values' in locals() and 'X_shap' in locals() and 'feature_names' in locals():
    print("SHAP data available, computing summaries and plots...")
    # Determine random seed and sample size used for SHAP (if available, else defaults)
    try:
        # These variables are set in the shap-load cell if SHAP file exists
        current_random_seed = random_seed
        current_shap_sample_size = shap_sample_size
    except NameError:
        # Fallback defaults
        current_random_seed = 42
        current_shap_sample_size = 100
        print(f"Using default random_seed={current_random_seed}, shap_sample_size={current_shap_sample_size}")
    else:
        print(f"Using SHAP-computed random_seed={current_random_seed}, shap_sample_size={current_shap_sample_size}")
    
    # Compute false positive and false negative SHAP summaries
    if fp_count > 0:
        fp_shap_summary = summarize_error_shap(
            shap_values, X_shap, feature_names,
            y, predictions,  # use full dataset labels/predictions
            error_type='false_positive',
            random_seed=current_random_seed,
            shap_sample_size=current_shap_sample_size
        )
        print(f"FP SHAP summary shape: {fp_shap_summary.shape}")
    else:
        fp_shap_summary = pd.DataFrame()
        print("No false positives found.")
    
    if fn_count > 0:
        fn_shap_summary = summarize_error_shap(
            shap_values, X_shap, feature_names,
            y, predictions,
            error_type='false_negative',
            random_seed=current_random_seed,
            shap_sample_size=current_shap_sample_size
        )
        print(f"FN SHAP summary shape: {fn_shap_summary.shape}")
    else:
        fn_shap_summary = pd.DataFrame()
        print("No false negatives found.")
    
    # Compute feature importance differences and value distributions
    try:
        feature_importance_diff = analyze_feature_importance_differences(
            shap_values, X_shap, feature_names,
            y, predictions,
            random_seed=current_random_seed,
            shap_sample_size=current_shap_sample_size
        )
        print(f"Feature importance diff shape: {feature_importance_diff.shape}")
    except Exception as e:
        print(f"Error computing feature importance differences: {e}")
        feature_importance_diff = pd.DataFrame()
    
    try:
        feature_value_dist = analyze_feature_value_distributions(
            shap_values, X_shap, feature_names,
            y, predictions,
            random_seed=current_random_seed,
            shap_sample_size=current_shap_sample_size
        )
        print(f"Feature value dist shape: {feature_value_dist.shape}")
    except Exception as e:
        print(f"Error computing feature value distributions: {e}")
        feature_value_dist = pd.DataFrame()
    
    # Generate plots for most confident false positive and false negative
    most_confident_fp_fig = None
    most_confident_fn_fig = None
    
    if fp_count > 0 and most_confident_fp_idx is not None:
        try:
            most_confident_fp_fig = plot_misclassification_shap(
                shap_values, X_shap, feature_names,
                y, predictions,
                observation_index=most_confident_fp_idx,
                save_path=None,  # we'll save via save_analysis_results later
                random_seed=current_random_seed,
                shap_sample_size=current_shap_sample_size
            )
            if most_confident_fp_fig is not None:
                print("FP plot generated successfully.")
            else:
                print("FP plot generation returned None.")
        except Exception as e:
            print(f"Error generating FP plot: {e}")
            import traceback
            traceback.print_exc()
    
    if fn_count > 0 and most_confident_fn_idx is not None:
        try:
            most_confident_fn_fig = plot_misclassification_shap(
                shap_values, X_shap, feature_names,
                y, predictions,
                observation_index=most_confident_fn_idx,
                save_path=None,
                random_seed=current_random_seed,
                shap_sample_size=current_shap_sample_size
            )
            if most_confident_fn_fig is not None:
                print("FN plot generated successfully.")
            else:
                print("FN plot generation returned None.")
        except Exception as e:
            print(f"Error generating FN plot: {e}")
            import traceback
            traceback.print_exc()
else:
    # Initialize empty dataframes/figures if SHAP not available
    fp_shap_summary = pd.DataFrame()
    fn_shap_summary = pd.DataFrame()
    feature_importance_diff = pd.DataFrame()
    feature_value_dist = pd.DataFrame()
    most_confident_fp_fig = None
    most_confident_fn_fig = None
    print("SHAP data not available for detailed analysis.")

In [ ]:
# Save analysis results
output_dir = EXPERIMENTS_DIR / latest_experiment.name / 'misclassification_analysis'
print(f"Saving analysis results to {output_dir}")

results_dict = {
    'misclassified_observations': misclassified_df if 'misclassified_df' in locals() else pd.DataFrame(),
    'false_positive_shap_summary': fp_shap_summary if 'fp_shap_summary' in locals() else pd.DataFrame(),
    'false_negative_shap_summary': fn_shap_summary if 'fn_shap_summary' in locals() else pd.DataFrame(),
    'feature_importance_differences': feature_importance_diff if 'feature_importance_diff' in locals() else pd.DataFrame(),
    'feature_value_distributions': feature_value_dist if 'feature_value_dist' in locals() else pd.DataFrame(),
}
# Add figures if they were generated
if 'most_confident_fp_fig' in locals() and most_confident_fp_fig is not None:
    results_dict['most_confident_fp_fig'] = most_confident_fp_fig
    print("Added FP figure to results_dict")
if 'most_confident_fn_fig' in locals() and most_confident_fn_fig is not None:
    results_dict['most_confident_fn_fig'] = most_confident_fn_fig
    print("Added FN figure to results_dict")
# Add directional vote error analysis if available (we haven't computed it, but we can compute if needed)
# For now, we'll skip unless we have it; but we can compute if we want.
# For now, we'll leave it out for brevity; but the function expects possibly a key.
# We'll compute it if we have time, but let's skip for now.

# Call save_analysis_results
saved_files = save_analysis_results(results_dict, output_dir)
print(f"Analysis results saved to {output_dir}")
print(f"Saved files: {saved_files}")

## 11. Misclassification Analysis Summary

This section prints a final summary of the misclassification analysis including counts of correct and incorrect predictions, and identifies the most important features for false positives and false negatives.

In [ ]:
# Print final summary
print("\n" + "="*60)
print("MISCLASSIFICATION ANALYSIS SUMMARY")
print("="*60)

if 'misclassified_df' in locals():
    total_obs = len(y)
    correct_pred = total_obs - len(misclassified_df)
    misclassified = len(misclassified_df)
    fp_count = len(misclassified_df[misclassified_df['error_type'] == 'false_positive']) if len(misclassified_df) > 0 else 0
    fn_count = len(misclassified_df[misclassified_df['error_type'] == 'false_negative']) if len(misclassified_df) > 0 else 0
    
    print(f"Training observations: {total_obs}")
    print(f"Correct predictions: {correct_pred}")
    print(f"Misclassified: {misclassified}")
    print(f"False positives: {fp_count}")
    print(f"False negatives: {fn_count}")
    
    print("\nMost important features for false positives:")
    if 'fp_shap_summary' in locals() and len(fp_shap_summary) > 0:
        top_fp_features = fp_shap_summary.head(5)[['feature', 'mean_abs_shap']].values
        for feature, importance in top_fp_features:
            print(f"  {feature}: {importance:.4f}")
    else:
        print("  No false positives found")
    
    print("\nMost important features for false negatives:")
    if 'fn_shap_summary' in locals() and len(fn_shap_summary) > 0:
        top_fn_features = fn_shap_summary.head(5)[['feature', 'mean_abs_shap']].values
        for feature, importance in top_fn_features:
            print(f"  {feature}: {importance:.4f}")
    else:
        print("  No false negatives found")
    
    print("\nFeature importance differences (incorrect vs correct):")
    if 'feature_importance_diff' in locals() and len(feature_importance_diff) > 0:
        top_diff_features = feature_importance_diff.head(5)[['feature', 'abs_mean_shap_diff']].values
        for feature, diff in top_diff_features:
            print(f"  {feature}: {diff:.4f}")
    else:
        print("  No feature importance differences calculated")
    
    print("\nFeature value distribution differences:")
    if 'feature_value_dist' in locals() and len(feature_value_dist) > 0:
        top_value_features = feature_value_dist.head(5)[['feature', 'mean_abs_diff']].values
        for feature, diff in top_value_features:
            print(f"  {feature}: {diff:.4f}")
    else:
        print("  No feature value distribution differences calculated")
        
else:
    print("Analysis not completed - SHAP values not available")

print("\n" + "="*60)
print("Analysis outputs saved to: experiments/20260814_084244/misclassification_analysis/")
print("="*60)

## 12. SHAP Visualizations

This section generates summary and dependence plots for SHAP values to provide deeper insights into feature contributions to model predictions.

In [ ]:
# Summary plot
plt.figure()
shap.summary_plot(shap_values, features=X_shap, feature_names=feature_names, show=False)
plt.title('SHAP Summary Plot')
plt.tight_layout()
plot_path = exp_dir / 'explanations' / 'shap_summary_notebook.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved summary plot to {plot_path}')

In [ ]:
# Dependence plots for top features
if 'shap_values' in locals() and 'X_shap' in locals() and 'feature_names' in locals():
    # Plot dependence plots for top 5 features
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": np.mean(np.abs(shap_values), axis=0)
    }).sort_values("importance", ascending=False)
    top_features = importance_df.head(5)["feature"].tolist()
    
    for feat in top_features:
        idx = feature_names.index(feat)
        # Additional safety check for each plot
        if shap_values.shape[1] == X_shap.shape[1]:
            plt.figure()
            shap.dependence_plot(idx, shap_values, features=X_shap, feature_names=feature_names, show=False)
            plt.title(f'SHAP Dependence: {feat}')
            plt.tight_layout()
            plot_path = exp_dir / 'explanations' / f'shap_dependence_{feat.replace(" ", "_").replace("/", "_")}.png'
            plt.savefig(plot_path, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'Saved dependence plot for {feat} to {plot_path}')
        else:
            print(f"Skipping dependence plot for {feat} due to dimension mismatch:")
            print(f"  SHAP values shape: {shap_values.shape}")
            print(f"  X_shap shape: {X_shap.shape}")
else:
    print("Skipping SHAP dependence plots - required variables not available")

## 13. Optuna Study Exploration

This section demonstrates how to analyze the Optuna study from the training process to understand hyperparameter optimization and model performance evolution.

In [ ]:


experiments_root = Path("../experiments")
# Find directories matching timestamp pattern
exp_folders = sorted(
    [p for p in experiments_root.iterdir() if p.is_dir() and re.match(r"\d{8}_\d{6}", p.name)],
    key=lambda p: p.name,
    reverse=True
)
if not exp_folders:
    raise FileNotFoundError("No experiment folders found in ../experiments")
latest_exp = exp_folders[0]
print(f"Using experiment folder: {latest_exp}")

study_path = latest_exp / "models" / "optuna_study.pkl"
study = joblib.load(study_path)
print(f"Loaded Optuna study with {len(study.trials)} trials.")


### 13.1 Study Overview

In [ ]:

best = study.best_trial
print(f"Best trial number: {best.number}")
print(f"Best value (competition score): {best.value:.5f}")
print("Best hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v}")


### 13.2 Trials DataFrame

In [ ]:

trials_df = study.trials_dataframe()
# Select a subset of columns for readability
cols = ["number", "value", "params_model_type", "params_learning_rate",
        "params_n_estimators", "params_max_depth", "params_subsample",
        "params_colsample_bytree", "state"]
# Keep only those that exist
existing_cols = [c for c in cols if c in trials_df.columns]
print(trials_df[existing_cols].head(10))
print(f"Total trials: {len(trials_df)}")


### 13.3 Hyperparameter Importance

In [ ]:

try:
    impt = optuna.importance.get_param_importances(study)
    if impt:
        # Sort
        sorted_impt = sorted(impt.items(), key=lambda x: x[1], reverse=True)
        params, importances = zip(*sorted_impt)
        plt.figure(figsize=(8,6))
        plt.barh(params, importances)
        plt.xlabel("Importance")
        plt.title("Hyperparameter Importance")
        plt.gca().invert_yaxis()  # highest on top
        plt.tight_layout()
        plt.show()
    else:
        print("Could not compute parameter importance (no trials with sufficient info).")
except Exception as e:
    print(f"Error plotting parameter importance: {e}")


### 13.4 Optimization History

In [ ]:

try:
    fig = vis.plot_optimization_history(study)
    fig.show()
except Exception as e:
    print(f"Error plotting optimization history: {e}")


### 13.5 Parallel Coordinate Plot

In [ ]:

try:
    fig = vis.plot_parallel_coordinate(study)
    fig.show()
except Exception as e:
    print(f"Error plotting parallel coordinate: {e}")


### 13.6 Slice Plot

In [ ]:

try:
    fig = vis.plot_slice(study)
    fig.show()
except Exception as e:
    print(f"Error plotting slice: {e}")


### 13.7 CV Score Evolution with Uncertainty Bands

In [ ]:
try:
    # Extract trials that completed successfully
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if trials:
        trial_numbers = [t.number for t in trials]
        cv_means = [t.user_attrs.get("cv_mean_score", 0) for t in trials]
        cv_stds = [t.user_attrs.get("cv_std_score", 0) for t in trials]
        
        # Create the plot
        plt.figure(figsize=(12, 6))
        
        # Plot mean CV score with error bands
        plt.plot(trial_numbers, cv_means, 'b-o', linewidth=2, markersize=4, label='Mean CV Score')
        plt.fill_between(trial_numbers, 
                         np.array(cv_means) - np.array(cv_stds),
                         np.array(cv_means) + np.array(cv_stds),
                         alpha=0.3, color='blue', label='±1 Standard Deviation')
        
        # Mark the best trial
        best_trial = study.best_trial
        plt.axvline(x=best_trial.number, color='red', linestyle='--', alpha=0.7, 
                   label=f'Best Trial ({best_trial.number})')
        plt.axhline(y=best_trial.value, color='red', linestyle=':', alpha=0.7,
                   label=f'Best Score ({best_trial.value:.4f})')
        
        plt.xlabel('Trial Number')
        plt.ylabel('Competition Score')
        plt.title('Cross-Validation Score Evolution During Optimization\n(Shaded area shows ±1 std dev across folds)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Print some summary statistics
        print(f"Best CV score: {best_trial.value:.4f} (trial {best_trial.number})")
        print(f"Final CV score: {cv_means[-1]:.4f} ± {cv_stds[-1]:.4f} (trial {trial_numbers[-1]})")
        print(f"Improvement: {cv_means[-1] - cv_means[0]:.4f} over {len(trials)} trials")
    else:
        print("No completed trials found for visualization.")
except Exception as e:
    print(f"Error plotting CV evolution: {e}")

### 13.8 Fold Score Distribution Across Trials

In [ ]:
try:
    # Extract trials that completed successfully
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if trials:
        # Determine number of folds from the first trial (assuming consistent CV setup)
        sample_trial = trials[0]
        fold_keys = [k for k in sample_trial.user_attrs.keys() if k.startswith('fold_') and k.endswith('_score')]
        n_folds = len(fold_keys)
        
        if n_folds > 0:
            # Prepare data for box plot: each box represents one fold across all trials
            fold_data = []
            fold_labels = []
            
            for i in range(n_folds):
                fold_scores = [t.user_attrs.get(f'fold_{i}_score', 0) for t in trials]
                fold_data.append(fold_scores)
                fold_labels.append(f'Fold {i+1}')
            
            # Create the plot
            plt.figure(figsize=(12, 6))
            
            box_plot = plt.boxplot(fold_data, tick_labels=fold_labels, patch_artist=True)
            
            # Customize box plot appearance
            for patch in box_plot['boxes']:
                patch.set_facecolor('lightblue')
                patch.set_alpha(0.7)
            
            # Add mean points for each fold
            fold_means = [np.mean(fold_data[i]) for i in range(n_folds)]
            plt.scatter(range(1, n_folds+1), fold_means, 
                       color='red', zorder=5, s=50, label='Mean per fold', 
                       edgecolors='darkred', linewidth=1)
            
            # Add overall statistics
            all_scores = [score for sublist in fold_data for score in sublist]
            overall_mean = np.mean(all_scores)
            overall_std = np.std(all_scores)
            
            plt.axhline(y=overall_mean, color='green', linestyle='-', alpha=0.7,
                       label=f'Overall Mean ({overall_mean:.4f})')
            plt.axhline(y=overall_mean + overall_std, color='green', linestyle='--', alpha=0.5,
                       label=f'+1 Std Dev ({overall_std:.4f})')
            plt.axhline(y=overall_mean - overall_std, color='green', linestyle='--', alpha=0.5,
                       label=f'-1 Std Dev ({overall_std:.4f})')
            
            plt.xlabel('CV Fold')
            plt.ylabel('Competition Score')
            plt.title(f'Distribution of Fold Scores Across {len(trials)} Optimization Trials\n'
                     f'(Shows consistency of performance across different data splits)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            # Print summary statistics
            print(f"Overall statistics across all folds and trials:")
            print(f"  Mean: {overall_mean:.4f}")
            print(f"  Std:  {overall_std:.4f}")
            print(f"  Min:  {np.min(all_scores):.4f}")
            print(f"  Max:  {np.max(all_scores):.4f}")
        else:
            print("No fold score data found in trials.")
    else:
        print("No completed trials found for visualization.")
except Exception as e:
    print(f"Error plotting fold distribution: {e}")

### 13.9 Optimization Progress Summary

## 14. Directional Vote Error Analysis

This section performs a targeted error-analysis of the current trained model to understand why it misclassifies some training samples, focusing on the directional-vote features that were shown to dominate the model in the SHAP analysis.

In [ ]:
# Analysis 1 — Summary statistics
# Split the training predictions into the four standard confusion-matrix groups:
# 1	True Positive (TP): true class = 1, predicted class = 1
# 2	True Negative (TN): true class = 0, predicted class = 0
# 3	False Positive (FP): true class = 0, predicted class = 1
# 4	False Negative (FN): true class = 1, predicted class = 0

# Define the six directional-vote features
directional_features = [
    "directional_vote_mean",
    "directional_vote_min",
    "directional_vote_max",
    "directional_vote_fraction_positive",
    "directional_vote_fraction_ge_2",
    "directional_vote_fraction_eq_4",
]

# Create masks for each group
tp_mask = (y == 1) & (predictions == 1)
tn_mask = (y == 0) & (predictions == 0)
fp_mask = (y == 0) & (predictions == 1)
fn_mask = (y == 1) & (predictions == 0)

# Create a dictionary to store results
results = []

# Get feature values for all samples (same as before)
if trainer is not None:
    X_features_all = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
    feature_names_list = list(trainer.feature_names) if hasattr(trainer, 'feature_names') else []
else:
    X_features_all = feature_engineer.transform(X, training=True).values
    feature_names_list = list(feature_engineer.get_feature_names_out()) if hasattr(feature_engineer, 'get_feature_names_out') else []

# Create masks for each group (same as before)
tp_mask = (y == 1) & (predictions == 1)
tn_mask = (y == 0) & (predictions == 0)
fp_mask = (y == 0) & (predictions == 1)
fn_mask = (y == 1) & (predictions == 0)

# Create a dictionary to store results
results = []

# For each feature and each group, calculate statistics
for feature in directional_features:
    if feature not in feature_names_list:
        print(f"Warning: Feature {feature} not found in feature names")
        continue
        
    # Get the index of this feature
    feature_idx = -1
    if hasattr(trainer, 'feature_names') and trainer.feature_names is not None:
        trainer_feature_names = list(trainer.feature_names)
        if feature in trainer_feature_names:
            feature_idx = trainer_feature_names.index(feature)
    elif hasattr(feature_engineer, 'get_feature_names_out') and feature_engineer.get_feature_names_out() is not None:
        fe_feature_names = list(feature_engineer.get_feature_names_out())
        if feature in fe_feature_names:
            feature_idx = fe_feature_names.index(feature)
    elif feature in feature_names:
        feature_idx = feature_names.index(feature)
    
    feature_values = X_features_all[:, feature_idx]
    
    # Calculate statistics for each group
    for group_name, mask in [('TP', tp_mask), ('TN', tn_mask), ('FP', fp_mask), ('FN', fn_mask)]:
        group_values = feature_values[mask]
        
        if len(group_values) > 0:
            group_stats = {
                'feature': feature,
                'prediction_group': group_name,
                'n': len(group_values),
                'mean': np.mean(group_values),
                'median': np.median(group_values),
                'std': np.std(group_values),
                'min': np.min(group_values),
                'q25': np.percentile(group_values, 25),
                'q75': np.percentile(group_values, 75),
                'max': np.max(group_values)
            }
        else:
            group_stats = {
                'feature': feature,
                'prediction_group': group_name,
                'n': 0,
                'mean': np.nan,
                'median': np.nan,
                'std': np.nan,
                'min': np.nan,
                'q25': np.nan,
                'q75': np.nan,
                'max': np.nan
            }
        results.append(group_stats)

# Convert to DataFrame and save
results_df = pd.DataFrame(results)
output_dir = EXPERIMENTS_DIR / latest_experiment.name / 'directional_vote_analysis'
output_dir.mkdir(exist_ok=True)
results_df.to_csv(output_dir / 'directional_vote_error_analysis.csv', index=False)

print("Summary statistics saved to:", output_dir / 'directional_vote_error_analysis.csv')
print("Results preview:")
print(results_df.head(10))


In [ ]:
# Analysis 2 — Distribution plots
# Create one figure for each directional-vote feature comparing the four groups:
# TP, TN, FP, FN

# Set up the plotting style
plt.style.use('default')
sns.set_palette("husl")

# Get feature values for all samples (same as in Analysis 1)
if trainer is not None:
    X_features_all = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
    feature_names_list = list(trainer.feature_names) if hasattr(trainer, 'feature_names') else []
else:
    X_features_all = feature_engineer.transform(X, training=True).values
    feature_names_list = list(feature_engineer.get_feature_names_out()) if hasattr(feature_engineer, 'get_feature_names_out') else []

# Create masks for each group (same as in Analysis 1)
tp_mask = (y == 1) & (predictions == 1)
tn_mask = (y == 0) & (predictions == 0)
fp_mask = (y == 0) & (predictions == 1)
fn_mask = (y == 1) & (predictions == 0)

# Group data for plotting
group_masks = {'TP': tp_mask, 'TN': tn_mask, 'FP': fp_mask, 'FN': fn_mask}
group_colors = {'TP': 'green', 'TN': 'blue', 'FP': 'red', 'FN': 'orange'}

# Create individual plots for each feature
for feature in directional_features:
    if feature not in feature_names_list:
        print(f"Warning: Feature {feature} not found in feature names")
        continue
        
    feature_idx = feature_names_list.index(feature)
    feature_values = X_features_all[:, feature_idx]
    
    # Create figure
    plt.figure(figsize=(12, 8))
    
    # Prepare data for boxplot
    box_data = []
    box_labels = []
    box_colors = []
    
    for group_name, mask in group_masks.items():
        group_values = feature_values[mask]
        if len(group_values) > 0:
            box_data.append(group_values)
            box_labels.append(group_name)
            box_colors.append(group_colors[group_name])
    
    # Create boxplot
    box_plot = plt.boxplot(box_data, labels=box_labels, patch_artist=True)
    
    # Customize box colors
    for patch, color in zip(box_plot['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add strip plot to show individual points (especially important for discrete features)
    for i, (group_name, mask) in enumerate(group_masks.items()):
        group_values = feature_values[mask]
        if len(group_values) > 0:
            # Add jitter to x-position for better visibility
            x_jitter = np.random.normal(i+1, 0.04, size=len(group_values))
            plt.scatter(x_jitter, group_values, alpha=0.6, s=20, color=group_colors[group_name])
    
    plt.title(f'Distribution of {feature} by Prediction Group\n'
              f'TP: True Positive, TN: True Negative, FP: False Positive, FN: False Negative')
    plt.ylabel(feature)
    plt.xlabel('Prediction Group')
    plt.grid(True, alpha=0.3)
    
    # Add legend
    legend_elements = [Patch(facecolor=group_colors[group], alpha=0.7, label=group) 
                       for group in group_masks.keys()]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    
    # Save plot
    plot_filename = output_dir / f'distribution_{feature.replace(" ", "_").replace("/", "_")}.png'
    plt.savefig(plot_filename, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Saved distribution plot for {feature} to {plot_filename}")

# Create a combined figure with all six features if practical
print("\nCreating combined figure with all six features...")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, feature in enumerate(directional_features):
    if feature not in feature_names_list:
        print(f"Warning: Feature {feature} not found in feature names")
        continue
        
    feature_idx = feature_names_list.index(feature)
    feature_values = X_features_all[:, feature_idx]
    
    ax = axes[idx]
    
    # Prepare data for boxplot
    box_data = []
    box_labels = []
    
    for group_name, mask in group_masks.items():
        group_values = feature_values[mask]
        if len(group_values) > 0:
            box_data.append(group_values)
            box_labels.append(group_name)
    
    # Create boxplot
    box_plot = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
    
    # Customize box colors
    for patch, group_name in zip(box_plot['boxes'], box_labels):
        patch.set_facecolor(group_colors[group_name])
        patch.set_alpha(0.7)
    
    ax.set_title(f'{feature}')
    ax.set_ylabel(feature)
    ax.set_xlabel('Prediction Group')
    ax.grid(True, alpha=0.3)
    
    # Add strip plot for individual points
    for i, (group_name, mask) in enumerate(group_masks.items()):
        group_values = feature_values[mask]
        if len(group_values) > 0:
            x_jitter = np.random.normal(i+1, 0.04, size=len(group_values))
            ax.scatter(x_jitter, group_values, alpha=0.6, s=15, color=group_colors[group_name])

# Add overall title and adjust layout
fig.suptitle('Directional Vote Features Distribution by Prediction Group\n'
             'TP: True Positive, TN: True Negative, FP: False Positive, FN: False Negative', 
             fontsize=16)
plt.tight_layout()

# Save combined figure
combined_plot_filename = output_dir / 'directional_vote_features_combined_distribution.png'
plt.savefig(combined_plot_filename, dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved combined distribution plot to {combined_plot_filename}")

In [ ]:
# Analysis 3 — Class-specific error analysis
# For each feature, calculate and report:
#   •	distribution for TN vs FP
#   •	distribution for TP vs FN
# Calculate quantitative measures of separation:
#   •	difference in medians
#   •	difference in means
#   •	Cohen's d
#   •	KS statistic

# Get feature values for all samples (same as before)
if trainer is not None:
    X_features_all = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
    feature_names_list = list(trainer.feature_names) if hasattr(trainer, 'feature_names') else []
else:
    X_features_all = feature_engineer.transform(X, training=True).values
    feature_names_list = list(feature_engineer.get_feature_names_out()) if hasattr(feature_engineer, 'get_feature_names_out') else []

# Create masks for each group (same as before)
tp_mask = (y == 1) & (predictions == 1)
tn_mask = (y == 0) & (predictions == 0)
fp_mask = (y == 0) & (predictions == 1)
fn_mask = (y == 1) & (predictions == 0)

# Prepare results for class-specific comparisons
comparisons = [
    ('TN_vs_FP', tn_mask, fp_mask),
    ('TP_vs_FN', tp_mask, fn_mask)
]

results = []

for feature in directional_features:
    if feature not in feature_names_list:
        print(f"Warning: Feature {feature} not found in feature names")
        continue
        
    feature_idx = feature_names_list.index(feature)
    feature_values = X_features_all[:, feature_idx]
    
    for comparison_name, mask1, mask2 in comparisons:
        group1_values = feature_values[mask1]
        group2_values = feature_values[mask2]
        
        if len(group1_values) > 0 and len(group2_values) > 0:
            # Basic statistics
            mean1, mean2 = np.mean(group1_values), np.mean(group2_values)
            median1, median2 = np.median(group1_values), np.median(group2_values)
            std1, std2 = np.std(group1_values, ddof=1), np.std(group2_values, ddof=1)
            
            # Difference in means
            mean_diff = mean2 - mean1  # group2 - group1
            
            # Difference in medians
            median_diff = median2 - median1
            
            # Cohen's d
            pooled_std = np.sqrt(((len(group1_values)-1)*std1**2 + (len(group2_values)-1)*std2**2) / (len(group1_values)+len(group2_values)-2))
            cohens_d = mean_diff / pooled_std if pooled_std != 0 else 0
            
            # KS statistic
            ks_stat, ks_pvalue = stats.ks_2samp(group1_values, group2_values)
            
            results.append({
                'feature': feature,
                'comparison': comparison_name,
                'group1_n': len(group1_values),
                'group2_n': len(group2_values),
                'group1_mean': mean1,
                'group2_mean': mean2,
                'group1_median': median1,
                'group2_median': median2,
                'group1_std': std1,
                'group2_std': std2,
                'mean_difference': mean_diff,
                'median_difference': median_diff,
                'cohens_d': cohens_d,
                'ks_statistic': ks_stat,
                'ks_pvalue': ks_pvalue
            })
        else:
            # Handle empty groups
            results.append({
                'feature': feature,
                'comparison': comparison_name,
                'group1_n': len(group1_values) if len(group1_values) > 0 else 0,
                'group2_n': len(group2_values) if len(group2_values) > 0 else 0,
                'group1_mean': np.nan,
                'group2_mean': np.nan,
                'group1_median': np.nan,
                'group2_median': np.nan,
                'group1_std': np.nan,
                'group2_std': np.nan,
                'mean_difference': np.nan,
                'median_difference': np.nan,
                'cohens_d': np.nan,
                'ks_statistic': np.nan,
                'ks_pvalue': np.nan
            })

# Convert to DataFrame and save
results_df = pd.DataFrame(results)
results_df.to_csv(output_dir / 'directional_vote_class_specific_analysis.csv', index=False)

print("Class-specific error analysis saved to:", output_dir / 'directional_vote_class_specific_analysis.csv')
print("\nResults preview:")
print(results_df.head(10))

# Print some key insights
print("\nKey Insights:")
print("=" * 50)
for comparison in ['TN_vs_FP', 'TP_vs_FN']:
    comp_results = results_df[results_df['comparison'] == comparison]
    if len(comp_results) > 0:
        print(f"\n{comparison}:")
        # Sort by absolute Cohen's d to find most discriminative features
        comp_results_sorted = comp_results.reindex(comp_results['cohens_d'].abs().sort_values(ascending=False).index)
        for _, row in comp_results_sorted.head(3).iterrows():
            print(f"  {row['feature']}: Cohen's d = {row['cohens_d']:.3f}, "
                  f"Mean diff = {row['mean_difference']:.3f}")

In [ ]:
# Analysis 4 — Probability of error as a function of directional vote
# For each directional-vote feature, investigate whether the probability of misclassification
# changes systematically with the feature value.
# Do this separately for:
#   •	true class = 0
#   •	true class = 1
# For continuous features such as directional_vote_mean, use sensible bins or quantiles.
# For discrete features such as the fraction features, use their actual possible values where practical.
# Create plots showing:
#   feature value → P(error | true class = 0)
#   feature value → P(error | true class = 1)

# Get feature values for all samples (same as before)
if trainer is not None:
    X_features_all = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
    feature_names_list = list(trainer.feature_names) if hasattr(trainer, 'feature_names') else []
else:
    X_features_all = feature_engineer.transform(X, training=True).values
    feature_names_list = list(feature_engineer.get_feature_names_out()) if hasattr(feature_engineer, 'get_feature_names_out') else []

# Create masks for true classes
true_class_0_mask = (y == 0)
true_class_1_mask = (y == 1)

# Create error masks
error_mask = (y != predictions)  # All errors
correct_mask = (y == predictions)  # All correct

# For each true class, we want P(error | feature value, true class)
# For true class = 0: errors are false positives
# For true class = 1: errors are false negatives

tp_mask = (y == 1) & (predictions == 1)
tn_mask = (y == 0) & (predictions == 0)
fp_mask = (y == 0) & (predictions == 1)  # Error when true class = 0
fn_mask = (y == 1) & (predictions == 0)  # Error when true class = 1

# Define which features are discrete/fractional (have limited possible values)
discrete_features = [
    "directional_vote_fraction_positive",
    "directional_vote_fraction_ge_2", 
    "directional_vote_fraction_eq_4"
]

# Define continuous features
continuous_features = [
    "directional_vote_mean",
    "directional_vote_min",
    "directional_vote_max"
]

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

# Create plots for each true class
for true_class_value, true_class_mask, error_type_name, error_mask_specific in [
    (0, true_class_0_mask, "False Positive (FP)", fp_mask),
    (1, true_class_1_mask, "False Negative (FN)", fn_mask)
]:
    print(f"\nAnalyzing P(error | feature value) for true class = {true_class_value} ({error_type_name})")
    
    # Determine number of subplots needed
    n_features = len(directional_features)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)
    
    fig.suptitle(f'Probability of Error vs Directional Vote Features\n'
                 f'True Class = {true_class_value} ({error_type_name})', fontsize=16)
    
    plot_index = 0
    
    for feature in directional_features:
        if feature not in feature_names_list:
            print(f"Warning: Feature {feature} not found in feature names")
            continue
            
        feature_idx = feature_names_list.index(feature)
        feature_values = X_features_all[:, feature_idx]
        
        # Get data for this true class only
        class_feature_values = feature_values[true_class_mask]
        class_error_values = error_mask_specific[true_class_mask]  # Errors within this true class
        
        # Calculate P(error | feature value)
        if feature in discrete_features:
            # For discrete features, use actual possible values
            # Get unique values and calculate error rate for each
            unique_vals = np.sort(np.unique(class_feature_values))
            error_rates = []
            
            for val in unique_vals:
                mask = (class_feature_values == val)
                if np.sum(mask) > 0:
                    error_rate = np.mean(class_error_values[mask])
                    error_rates.append(error_rate)
                else:
                    error_rates.append(np.nan)
            
            # Plot as bar plot or scatter plot
            row_idx = plot_index // n_cols
            col_idx = plot_index % n_cols
            ax = axes[row_idx, col_idx]
            
            bars = ax.bar(range(len(unique_vals)), error_rates, alpha=0.7, color='skyblue', edgecolor='navy')
            ax.set_xticks(range(len(unique_vals)))
            ax.set_xticklabels([f'{val:.3f}' for val in unique_vals], rotation=45)
            ax.set_xlabel(f'{feature} value')
            ax.set_ylabel('P(Error | feature value)')
            ax.set_title(f'{feature}')
            ax.grid(True, alpha=0.3)
            
            # Add value labels on bars
            for i, (rate, val) in enumerate(zip(error_rates, unique_vals)):
                if not np.isnan(rate):
                    ax.text(i, rate + 0.01, f'{rate:.3f}', ha='center', va='bottom', fontsize=8)
                    
        else:
            # For continuous features, use quantiles or bins
            row_idx = plot_index // n_cols
            col_idx = plot_index % n_cols
            ax = axes[row_idx, col_idx]
            
            # Use quantiles to create bins
            n_bins = min(10, len(np.unique(class_feature_values)) // 5)  # Adaptive number of bins
            n_bins = max(5, n_bins)  # At least 5 bins
            
            # Create bins based on quantiles
            bins = np.percentile(class_feature_values, np.linspace(0, 100, n_bins + 1))
            # Ensure unique bins
            bins = np.unique(bins)
            
            # Assign each value to a bin
            bin_indices = np.digitize(class_feature_values, bins[:-1])  # Values >= bins[i] and < bins[i+1] go to bin i
            bin_indices = np.clip(bin_indices, 0, len(bins)-2)  # Ensure valid indices
            
            # Calculate error rate for each bin
            bin_centers = []
            bin_error_rates = []
            bin_counts = []
            
            for bin_idx in range(len(bins)-1):
                bin_mask = (bin_indices == bin_idx)
                if np.sum(bin_mask) > 0:
                    bin_center = (bins[bin_idx] + bins[bin_idx+1]) / 2
                    bin_error_rate = np.mean(class_error_values[bin_mask])
                    bin_count = np.sum(bin_mask)
                    
                    bin_centers.append(bin_center)
                    bin_error_rates.append(bin_error_rate)
                    bin_counts.append(bin_count)
            
            # Plot as line plot with error bars showing uncertainty
            if len(bin_centers) > 0:
                # Calculate binomial proportion confidence interval (Wilson score interval)
                ci_lower = []
                ci_upper = []
                for rate, count in zip(bin_error_rates, bin_counts):
                    if count > 0:
                        ci = smp.proportion_confint(count * rate, count, alpha=0.05, method='wilson')
                        ci_lower.append(ci[0])
                        ci_upper.append(ci[1])
                    else:
                        ci_lower.append(0)
                        ci_upper.append(0)
                
                ax.plot(bin_centers, bin_error_rates, 'o-', linewidth=2, markersize=6, color='darkblue')
                ax.fill_between(bin_centers, ci_lower, ci_upper, alpha=0.3, color='lightblue')
                ax.set_xlabel(f'{feature} value')
                ax.set_ylabel('P(Error | feature value)')
                ax.set_title(f'{feature}')
                ax.grid(True, alpha=0.3)
                
                # Add sample size annotations
                for i, (center, rate, count) in enumerate(zip(bin_centers, bin_error_rates, bin_counts)):
                    ax.text(center, rate + 0.02, f'n={count}', ha='center', va='bottom', fontsize=8, alpha=0.7)
        
        plot_index += 1
    
    # Hide unused subplots
    total_plots = n_rows * n_cols
    for i in range(plot_index, total_plots):
        row_idx = i // n_cols
        col_idx = i % n_cols
        axes[row_idx, col_idx].set_visible(False)
    
    plt.tight_layout()
    
    # Save plot
    plot_filename = output_dir / f'prob_error_vs_features_trueclass{true_class_value}.png'
    plt.savefig(plot_filename, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Saved probability of error plot for true class {true_class_value} to {plot_filename}")

# Also create a combined plot showing both true classes on the same axes for comparison
print("\nCreating combined plot for both true classes...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(directional_features):
    if feature not in feature_names_list:
        print(f"Warning: Feature {feature} not found in feature names")
        continue
        
    feature_idx = feature_names_list.index(feature)
    feature_values = X_features_all[:, feature_idx]
    
    ax = axes[idx]
    
    # Plot for both true classes
    for true_class_value, color, linestyle, label in [
        (0, 'blue', '-', 'True Class = 0'),
        (1, 'red', '--', 'True Class = 1')
    ]:
        true_class_mask = (y == true_class_value)
        error_mask_specific = fp_mask if true_class_value == 0 else fn_mask
        
        # Get data for this true class
        class_feature_values = feature_values[true_class_mask]
        class_error_values = error_mask_specific[true_class_mask]
        
        if feature in discrete_features:
            # For discrete features
            unique_vals = np.sort(np.unique(class_feature_values))
            error_rates = []
            counts = []
            
            for val in unique_vals:
                mask = (class_feature_values == val)
                if np.sum(mask) > 0:
                    error_rate = np.mean(class_error_values[mask])
                    count = np.sum(mask)
                    error_rates.append(error_rate)
                    counts.append(count)
                else:
                    error_rates.append(np.nan)
                    counts.append(0)
            
            # Plot as lines with markers
            valid_mask = ~np.isnan(error_rates)
            if np.sum(valid_mask) > 0:
                ax.plot(np.array(unique_vals)[valid_mask], np.array(error_rates)[valid_mask], 
                       marker='o', linestyle=linestyle, color=color, label=label, linewidth=2, markersize=4)
                
                # Add sample size annotations for a few points
                for i in range(0, len(unique_vals), max(1, len(unique_vals)//5)):  # Show every nth point
                    if valid_mask[i]:
                        ax.text(unique_vals[i], error_rates[i] + 0.02, f'n={counts[i]}', 
                               ha='center', va='bottom', fontsize=7, alpha=0.7, color=color)
        else:
            # For continuous features - use smoothing or binning
            # Use quantile-based binning for both classes consistently
            all_class_values = feature_values[true_class_mask]
            if len(all_class_values) > 0:
                n_bins = min(8, len(np.unique(all_class_values)) // 3)
                n_bins = max(3, n_bins)
                
                bins = np.percentile(all_class_values, np.linspace(0, 100, n_bins + 1))
                bins = np.unique(bins)
                
                if len(bins) > 1:
                    bin_indices = np.digitize(all_class_values, bins[:-1])
                    bin_indices = np.clip(bin_indices, 0, len(bins)-2)
                    
                    bin_centers = []
                    bin_error_rates = []
                    
                    for bin_idx in range(len(bins)-1):
                        bin_mask = (bin_indices == bin_idx)
                        if np.sum(bin_mask) > 0:
                            bin_center = (bins[bin_idx] + bins[bin_idx+1]) / 2
                            bin_error_rate = np.mean(class_error_values[bin_mask])
                            bin_centers.append(bin_center)
                            bin_error_rates.append(bin_error_rate)
                    
                    if len(bin_centers) > 0:
                        ax.plot(bin_centers, bin_error_rates, marker='s', linestyle=linestyle, 
                               color=color, label=label, linewidth=2, markersize=4)
    
    ax.set_xlabel(feature)
    ax.set_ylabel('P(Error | feature value)')
    ax.set_title(f'{feature}')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Probability of Error vs Directional Vote Features\n'
             'Solid line: True Class = 0, Dashed line: True Class = 1', fontsize=16)
plt.tight_layout()

# Save combined plot
combined_plot_filename = output_dir / 'prob_error_vs_features_both_classes.png'
plt.savefig(combined_plot_filename, dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved combined probability of error plot to {combined_plot_filename}")

In [ ]:
# Analysis 5 — Prediction confidence
# Include the model's predicted probability in the error analysis.
# For each TP/TN/FP/FN group, report:
#   •	number of samples
#   •	mean predicted probability
#   •	median predicted probability
#   •	distribution of predicted probabilities
# For FP and FN specifically, determine whether the errors tend to be:
#   •	high-confidence errors
#   •	low-confidence errors near the decision boundary
# Create a probability-distribution plot for TP/TN/FP/FN.


# We already have:
# predictions: predicted class (0 or 1)
# probabilities: predicted probability for class 1
# y: true class (0 or 1)

# Define groups (same as before)
tp_mask = (y == 1) & (predictions == 1)  # True Positive
tn_mask = (y == 0) & (predictions == 0)  # True Negative
fp_mask = (y == 0) & (predictions == 1)  # False Positive
fn_mask = (y == 1) & (predictions == 0)  # False Negative

group_masks = {
    'TP': tp_mask,
    'TN': tn_mask,
    'FP': fp_mask,
    'FN': fn_mask
}

group_labels = {
    'TP': 'True Positive',
    'TN': 'True Negative', 
    'FP': 'False Positive',
    'FN': 'False Negative'
}

group_colors = {
    'TP': 'green',
    'TN': 'blue',
    'FP': 'red',
    'FN': 'orange'
}

# Calculate statistics for each group
results = []

for group_name, mask in group_masks.items():
    group_probabilities = probabilities[mask]
    
    if len(group_probabilities) > 0:
        conf_stats = {
            'group': group_name,
            'group_label': group_labels[group_name],
            'n_samples': len(group_probabilities),
            'mean_probability': np.mean(group_probabilities),
            'median_probability': np.median(group_probabilities),
            'std_probability': np.std(group_probabilities),
            'min_probability': np.min(group_probabilities),
            'q25_probability': np.percentile(group_probabilities, 25),
            'q75_probability': np.percentile(group_probabilities, 75),
            'max_probability': np.max(group_probabilities)
        }
    else:
        conf_stats = {
            'group': group_name,
            'group_label': group_labels[group_name],
            'n_samples': 0,
            'mean_probability': np.nan,
            'median_probability': np.nan,
            'std_probability': np.nan,
            'min_probability': np.nan,
            'q25_probability': np.nan,
            'q75_probability': np.nan,
            'max_probability': np.nan
        }
    results.append(conf_stats)

# Convert to DataFrame and save
confidence_df = pd.DataFrame(results)
confidence_df.to_csv(output_dir / 'prediction_confidence_analysis.csv', index=False)

print("Prediction confidence analysis saved to:", output_dir / 'prediction_confidence_analysis.csv')
print("\nConfidence statistics:")
print(confidence_df[['group', 'n_samples', 'mean_probability', 'median_probability']].to_string(index=False))

# Determine whether FP and FN errors are high-confidence or low-confidence
print("\nError Analysis:")
print("=" * 50)
fp_stats = confidence_df[confidence_df['group'] == 'FP'].iloc[0] if len(confidence_df[confidence_df['group'] == 'FP']) > 0 else None
fn_stats = confidence_df[confidence_df['group'] == 'FN'].iloc[0] if len(confidence_df[confidence_df['group'] == 'FN']) > 0 else None

if fp_stats is not None:
    mean_fp_prob = fp_stats['mean_probability']
    print(f"False Positives (true=0, pred=1):")
    print(f"  Mean predicted probability: {mean_fp_prob:.3f}")
    if mean_fp_prob > 0.7:
        print(f"  → HIGH-CONFIDENCE errors (model strongly believed these were class 1)")
    elif mean_fp_prob < 0.3:
        print(f"  → LOW-CONFIDENCE errors (model weakly believed these were class 1)")
    else:
        print(f"  → MEDIUM-CONFIDENCE errors (model uncertain about these predictions)")
    print(f"  Interpretation: Model predicted class 1 with probability {mean_fp_prob:.3f} but true class was 0")

if fn_stats is not None:
    mean_fn_prob = fn_stats['mean_probability']
    print(f"\nFalse Negatives (true=1, pred=0):")
    print(f"  Mean predicted probability: {mean_fn_prob:.3f}")
    if mean_fn_prob > 0.7:
        print(f"  → HIGH-CONFIDENCE errors (model strongly believed these were class 1 but predicted 0)")
    elif mean_fn_prob < 0.3:
        print(f"  → LOW-CONFIDENCE errors (model weakly believed these were class 1)")
    else:
        print(f"  → MEDIUM-CONFIDENCE errors (model uncertain about these predictions)")
    print(f"  Interpretation: Model predicted class 0 (probability {1-mean_fn_prob:.3f} for class 1) but true class was 1")

# Create probability-distribution plot for TP/TN/FP/FN
plt.figure(figsize=(14, 8))

# Plot histograms for each group
for group_name, mask in group_masks.items():
    group_probabilities = probabilities[mask]
    if len(group_probabilities) > 0:
        plt.hist(group_probabilities, bins=30, alpha=0.7, label=group_labels[group_name], 
                 color=group_colors[group_name], density=True, edgecolor='black', linewidth=0.5)

# Add vertical line at decision threshold (0.5)
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')

# Customize plot
plt.xlabel('Predicted Probability of Class 1', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Distribution of Predicted Probabilities by Prediction Group\n'
          'TP: True Positive, TN: True Negative, FP: False Positive, FN: False Negative', 
          fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Add text box with key insights
if fp_stats is not None and fn_stats is not None:
    insights = []
    if fp_stats['mean_probability'] > 0.7:
        insights.append("FP: High-confidence errors")
    elif fp_stats['mean_probability'] < 0.3:
        insights.append("FP: Low-confidence errors")
    else:
        insights.append("FP: Medium-confidence errors")
        
    if fn_stats['mean_probability'] > 0.7:
        insights.append("FN: High-confidence errors")
    elif fn_stats['mean_probability'] < 0.3:
        insights.append("FN: Low-confidence errors")
    else:
        insights.append("FN: Medium-confidence errors")
    
    insights_text = "\n".join(insights)
    plt.text(0.02, 0.98, insights_text, transform=plt.gca().transAxes, 
             fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()

# Save plot
confidence_plot_filename = output_dir / 'prediction_confidence_distribution.png'
plt.savefig(confidence_plot_filename, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved prediction confidence distribution plot to {confidence_plot_filename}")

# Additional analysis: Calculate proportion of high/medium/low confidence errors
print("\nError Confidence Breakdown:")
print("=" * 50)
for group_name in ['FP', 'FN']:
    error_stat = confidence_df[confidence_df['group'] == group_name].iloc[0] if len(confidence_df[confidence_df['group'] == group_name]) > 0 else None
    if error_stat is not None:
        mean_prob = error_stat['mean_probability']
        n_samples = error_stat['n_samples']
        if n_samples > 0:
            if group_name == 'FP':
                # For FP, high probability means high confidence in wrong prediction
                if mean_prob > 0.7:
                    conf_level = "High"
                elif mean_prob < 0.3:
                    conf_level = "Low"
                else:
                    conf_level = "Medium"
            else:  # FN
                # For FN, low probability means high confidence in wrong prediction (since they should be class 1)
                inv_prob = 1 - mean_prob  # Probability of being class 1
                if inv_prob > 0.7:
                    conf_level = "High"
                elif inv_prob < 0.3:
                    conf_level = "Low"
                else:
                    conf_level = "Medium"
            print(f"{group_name}s (n={n_samples}): {conf_level} confidence errors (mean prob = {mean_prob:.3f})")

In [ ]:
# Analysis 6 — Individual misclassification examples
# Create a CSV containing every FP and FN with:
#   •	sample/index ID
#   •	true class
#   •	predicted class
#   •	predicted probability
#   •	all six directional-vote features
#   •	the SHAP values for those six features, if available

import numpy as np
import pandas as pd

# Get feature values for all samples (same as before)
if trainer is not None:
    X_features_all = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
    feature_names_list = list(trainer.feature_names) if hasattr(trainer, 'feature_names') else []
else:
    X_features_all = feature_engineer.transform(X, training=True).values
    feature_names_list = list(feature_engineer.get_feature_names_out()) if hasattr(feature_engineer, 'get_feature_names_out') else []

# Get the six directional-vote features
directional_features = [
    "directional_vote_mean",
    "directional_vote_min",
    "directional_vote_max",
    "directional_vote_fraction_positive",
    "directional_vote_fraction_ge_2",
    "directional_vote_fraction_eq_4",
]

# Check which directional features are available
available_directional_features = [f for f in directional_features if f in feature_names_list]
missing_directional_features = [f for f in directional_features if f not in feature_names_list]

if missing_directional_features:
    print(f"Warning: The following directional-vote features are not available: {missing_directional_features}")

print(f"Available directional-vote features: {available_directional_features}")

# Get indices for these features
directional_feature_indices = [feature_names_list.index(f) for f in available_directional_features]
directional_feature_values = X_features_all[:, directional_feature_indices]

# Get SHAP values if available
shap_values_available = 'shap_values' in locals() and 'X_shap' in locals() and 'feature_names' in locals()
if shap_values_available:
    print("SHAP values are available - will include SHAP values for directional-vote features in output")
    # Get SHAP values for directional-vote features
    shap_feature_names = list(feature_names)
    shap_directional_indices = [shap_feature_names.index(f) for f in available_directional_features if f in shap_feature_names]
    
    # Handle case where SHAP was computed on subset
    n_samples = len(y)
    n_shap = shap_values.shape[0]
    if n_shap == n_samples:
        # SHAP values were computed on full dataset
        shap_indices_full = np.arange(n_samples)
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        try:
            current_random_seed = random_seed
            current_shap_sample_size = shap_sample_size
        except NameError:
            # Fallback defaults
            current_random_seed = 42
            current_shap_sample_size = 100
        
        if current_shap_sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(current_random_seed)
            shap_indices_full = rng.choice(n_samples, size=current_shap_sample_size, replace=False)
        else:
            # current_shap_sample_size >= n_samples, effectively full dataset
            shap_indices_full = np.arange(n_samples)
    
    # Map from full dataset indices to SHAP subset indices
    shap_index_map = {full_idx: subset_idx for subset_idx, full_idx in enumerate(shap_indices_full)}
else:
    print("SHAP values not available - will not include SHAP values in output")
    shap_index_map = {}

# Create masks for misclassified observations
fp_mask = (y == 0) & (predictions == 1)  # False Positive
fn_mask = (y == 1) & (predictions == 0)  # False Negative
misclassified_mask = fp_mask | fn_mask

# Get indices of misclassified observations
misclassified_indices = np.where(misclassified_mask)[0]
fp_indices = np.where(fp_mask)[0]
fn_indices = np.where(fn_mask)[0]

print(f"Total misclassified observations: {len(misclassified_indices)}")
print(f"  False Positives: {len(fp_indices)}")
print(f"  False Negatives: {len(fn_indices)}")

# Prepare data for CSV
rows = []

for idx in misclassified_indices:
    # Basic information
    row = {
        'sample_index': idx,
        'true_class': int(y[idx]),
        'predicted_class': int(predictions[idx]),
        'predicted_probability': float(probabilities[idx])
    }
    
    # Add directional-vote feature values
    for i, feature in enumerate(available_directional_features):
        row[f'{feature}_value'] = float(directional_feature_values[idx, i])
    
    # Add SHAP values if available
    if shap_values_available and idx in shap_index_map:
        shap_idx = shap_index_map[idx]
        for i, feature in enumerate(available_directional_features):
            if i < len(shap_directional_indices):  # Make sure we have SHAP for this feature
                shap_feature_idx = shap_directional_indices[i]
                if shap_idx < shap_values.shape[0] and shap_feature_idx < shap_values.shape[1]:  # Bounds check
                    row[f'{feature}_shap_value'] = float(shap_values[shap_idx, shap_feature_idx])
                else:
                    row[f'{feature}_shap_value'] = np.nan
            else:
                row[f'{feature}_shap_value'] = np.nan
    elif shap_values_available:
        # Index not in SHAP subset
        for i, feature in enumerate(available_directional_features):
            row[f'{feature}_shap_value'] = np.nan
    
    rows.append(row)

# Convert to DataFrame
misclassified_df = pd.DataFrame(rows)

# Reorder columns for clarity
base_cols = ['sample_index', 'true_class', 'predicted_class', 'predicted_probability']
directional_value_cols = [f'{f}_value' for f in available_directional_features]
if shap_values_available:
    directional_shap_cols = [f'{f}_shap_value' for f in available_directional_features]
    column_order = base_cols + directional_value_cols + directional_shap_cols
else:
    column_order = base_cols + directional_value_cols

misclassified_df = misclassified_df[column_order]

# Save to CSV
misclassified_csv_path = output_dir / 'individual_misclassification_examples.csv'
misclassified_df.to_csv(misclassified_csv_path, index=False)

print(f"\nIndividual misclassification examples saved to: {misclassified_csv_path}")
print(f"Shape: {misclassified_df.shape}")

# Display first few rows
print("\nFirst 5 rows of misclassified examples:")
print(misclassified_df.head().to_string(index=False))

# Display summary by error type
print("\nSummary by error type:")
error_type_col = []
for idx in misclassified_indices:
    if y[idx] == 0 and predictions[idx] == 1:
        error_type_col.append('FP')
    elif y[idx] == 1 and predictions[idx] == 0:
        error_type_col.append('FN')
    else:
        error_type_col.append('Other')

misclassified_df['error_type'] = error_type_col
error_summary = misclassified_df['error_type'].value_counts()
print(error_summary.to_string())

# Show some example cases with highest confidence errors
print("\nHighest confidence False Positives (model most sure they were class 1 but were actually class 0):")
if len(fp_indices) > 0:
    fp_probabilities = probabilities[fp_indices]
    fp_sorted_idx = fp_indices[np.argsort(fp_probabilities)[::-1]]  # Descending order
    top_n = min(5, len(fp_sorted_idx))
    for i in range(top_n):
        idx = fp_sorted_idx[i]
        print(f"  Index {idx}: True={y[idx]}, Pred={predictions[idx]}, Prob={probabilities[idx]:.3f}")

print("\nHighest confidence False Negatives (model least sure they were class 1 but were actually class 1):")
if len(fn_indices) > 0:
    fn_probabilities = probabilities[fn_indices]
    fn_sorted_idx = fn_indices[np.argsort(fn_probabilities)]  # Ascending order (lowest probability first)
    top_n = min(5, len(fn_sorted_idx))
    for i in range(top_n):
        idx = fn_sorted_idx[i]
        print(f"  Index {idx}: True={y[idx]}, Pred={predictions[idx]}, Prob={probabilities[idx]:.3f}")

## 15. Directional Vote Error Analysis Summary

This section provides a concise automatically generated summary of the directional vote error analysis.

In [ ]:
# Generate concise automatically generated summary
print("\n" + "="*80)
print("DIRECTIONAL VOTE ERROR ANALYSIS SUMMARY")
print("="*80)

# Load the results from our analyses if available, otherwise compute key metrics
try:
    # Try to load saved results
    summary_stats = pd.read_csv(output_dir / 'directional_vote_error_analysis.csv')
    class_specific = pd.read_csv(output_dir / 'directional_vote_class_specific_analysis.csv')
    confidence_stats = pd.read_csv(output_dir / 'prediction_confidence_analysis.csv')
    misclassified_examples = pd.read_csv(output_dir / 'individual_misclassification_examples.csv')
    
    print("✓ Loaded analysis results from saved files")
    
except Exception as e:
    print(f"⚠ Could not load saved results: {e}")
    print("Computing summary metrics directly...")
    
    # Compute key metrics directly
    # Define groups
    tp_mask = (y == 1) & (predictions == 1)
    tn_mask = (y == 0) & (predictions == 0)
    fp_mask = (y == 0) & (predictions == 1)
    fn_mask = (y == 1) & (predictions == 0)
    
    # Get feature values
    if trainer is not None:
        X_features_all = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
        feature_names_list = list(trainer.feature_names) if hasattr(trainer, 'feature_names') else []
    else:
        X_features_all = feature_engineer.transform(X, training=True).values
        feature_names_list = list(feature_engineer.get_feature_names_out()) if hasattr(feature_engineer, 'get_feature_names_out') else []
    
    directional_features = [
        "directional_vote_mean",
        "directional_vote_min",
        "directional_vote_max",
        "directional_vote_fraction_positive",
        "directional_vote_fraction_ge_2",
        "directional_vote_fraction_eq_4",
    ]
    
    # Filter to available features
    available_directional_features = [f for f in directional_features if f in feature_names_list]
    
    if available_directional_features:
        feature_indices = [feature_names_list.index(f) for f in available_directional_features]
        feature_values = X_features_all[:, feature_indices]
    else:
        feature_values = np.empty((len(y), 0))
    
    # Create summary stats dataframe manually
    summary_data = []
    for feature in available_directional_features:
        feature_idx = feature_names_list.index(feature)
        feat_vals = feature_values[:, feature_idx]
        for group_name, mask in [('TP', tp_mask), ('TN', tn_mask), ('FP', fp_mask), ('FN', fn_mask)]:
            group_vals = feat_vals[mask]
            if len(group_vals) > 0:
                summary_data.append({
                    'feature': feature,
                    'prediction_group': group_name,
                    'n': len(group_vals),
                    'mean': np.mean(group_vals),
                    'median': np.median(group_vals),
                    'std': np.std(group_vals),
                    'min': np.min(group_vals),
                    'q25': np.percentile(group_vals, 25),
                    'q75': np.percentile(group_vals, 75),
                    'max': np.max(group_vals)
                })
    
    summary_stats = pd.DataFrame(summary_data) if summary_data else pd.DataFrame()
    
    # Compute class-specific analysis
    class_specific_data = []
    comparisons = [('TN_vs_FP', tn_mask, fp_mask), ('TP_vs_FN', tp_mask, fn_mask)]
    for feature in available_directional_features:
        feature_idx = feature_names_list.index(feature)
        feat_vals = feature_values[:, feature_idx]
        for comparison_name, mask1, mask2 in comparisons:
            group1_vals = feat_vals[mask1]
            group2_vals = feat_vals[mask2]
            if len(group1_vals) > 0 and len(group2_vals) > 0:
                mean1, mean2 = np.mean(group1_vals), np.mean(group2_vals)
                median1, median2 = np.median(group1_vals), np.median(group2_vals)
                std1, std2 = np.std(group1_vals, ddof=1), np.std(group2_vals, ddof=1)
                pooled_std = np.sqrt(((len(group1_vals)-1)*std1**2 + (len(group2_vals)-1)*std2**2) / (len(group1_vals)+len(group2_vals)-2))
                cohens_d = (mean2 - mean1) / pooled_std if pooled_std != 0 else 0
                ks_stat, _ = stats.ks_2samp(group1_vals, group2_vals)
                
                class_specific_data.append({
                    'feature': feature,
                    'comparison': comparison_name,
                    'group1_n': len(group1_vals),
                    'group2_n': len(group2_vals),
                    'group1_mean': mean1,
                    'group2_mean': mean2,
                    'group1_median': median1,
                    'group2_median': median2,
                    'mean_difference': mean2 - mean1,
                    'median_difference': median2 - median1,
                    'cohens_d': cohens_d,
                    'ks_statistic': ks_stat
                })
    
    class_specific = pd.DataFrame(class_specific_data) if class_specific_data else pd.DataFrame()
    
    # Compute confidence stats
    confidence_data = []
    for group_name, mask in [('TP', tp_mask), ('TN', tn_mask), ('FP', fp_mask), ('FN', fn_mask)]:
        group_probs = probabilities[mask]
        if len(group_probs) > 0:
            confidence_data.append({
                'group': group_name,
                'group_label': {'TP': 'True Positive', 'TN': 'True Negative', 'FP': 'False Positive', 'FN': 'False Negative'}[group_name],
                'n_samples': len(group_probs),
                'mean_probability': np.mean(group_probs),
                'median_probability': np.median(group_probs),
                'std_probability': np.std(group_probs),
                'min_probability': np.min(group_probs),
                'q25_probability': np.percentile(group_probs, 25),
                'q75_probability': np.percentile(group_probs, 75),
                'max_probability': np.max(group_probs)
            })
    
    confidence_stats = pd.DataFrame(confidence_data) if confidence_data else pd.DataFrame()
    
    # Create misclassified examples dataframe
    misclassified_mask = (y != predictions)
    misclassified_indices = np.where(misclassified_mask)[0]
    
    misclassified_data = []
    for idx in misclassified_indices:
        row = {
            'sample_index': idx,
            'true_class': int(y[idx]),
            'predicted_class': int(predictions[idx]),
            'predicted_probability': float(probabilities[idx])
        }
        for i, feature in enumerate(available_directional_features):
            if i < len(feature_indices):
                row[f'{feature}_value'] = float(feature_values[idx, i])
        misclassified_data.append(row)
    
    misclassified_examples = pd.DataFrame(misclassified_data) if misclassified_data else pd.DataFrame()

# Now generate the summary based on the loaded/computed data
print("\n1. Which directional-vote features best separate TP/TN from FP/FN.")
print("-" * 60)

if not class_specific.empty:
    # Look at TN_vs_FP and TP_vs_FN comparisons
    tn_fp_results = class_specific[class_specific['comparison'] == 'TN_vs_FP'] if 'comparison' in class_specific.columns else pd.DataFrame()
    tp_fn_results = class_specific[class_specific['comparison'] == 'TP_vs_FN'] if 'comparison' in class_specific.columns else pd.DataFrame()
    
    # Combine both comparisons to find features that separate well in both
    if not tn_fp_results.empty and not tp_fn_results.empty:
        # For TN_vs_FP: we want features where TN and FP are well separated
        # For TP_vs_FN: we want features where TP and FN are well separated
        
        # Calculate average absolute Cohen's d across both comparisons for each feature
        feature_separation = {}
        for feature in available_directional_features:
            tn_fp_d = tn_fp_results[tn_fp_results['feature'] == feature]['cohens_d'].iloc[0] if not tn_fp_results.empty and feature in tn_fp_results['feature'].values else 0
            tp_fn_d = tp_fn_results[tp_fn_results['feature'] == feature]['cohens_d'].iloc[0] if not tp_fn_results.empty and feature in tp_fn_results['feature'].values else 0
            avg_abs_d = (abs(tn_fp_d) + abs(tp_fn_d)) / 2
            feature_separation[feature] = avg_abs_d
        
        # Sort by separation ability
        sorted_features = sorted(feature_separation.items(), key=lambda x: x[1], reverse=True)
        
        print("Features ranked by ability to separate correct from incorrect predictions:")
        for feature, score in sorted_features:
            print(f"  {feature}: {score:.3f} (average |Cohen's d|)")
        
        # Top 3 features
        top_3 = sorted_features[:3]
        print(f"\nTop 3 features for separating TP/TN from FP/FN:")
        for feature, score in top_3:
            print(f"  {feature}")
    else:
        print("⚠ Insufficient data for class-specific separation analysis")
else:
    print("⚠ No directional-vote features found for analysis")

print("\n2. Whether FP and FN appear to occupy distinct regions of directional-vote space.")
print("-" * 60)

if not summary_stats.empty:
    # Compare the distributions of FP vs FN for each feature
    fp_fn_distinct = {}
    for feature in available_directional_features:
        # Get FP and FN statistics
        fp_stats = summary_stats[(summary_stats['feature'] == feature) & (summary_stats['prediction_group'] == 'FP')]
        fn_stats = summary_stats[(summary_stats['feature'] == feature) & (summary_stats['prediction_group'] == 'FN')]
        
        if not fp_stats.empty and not fn_stats.empty:
            fp_mean = fp_stats['mean'].iloc[0]
            fn_mean = fn_stats['mean'].iloc[0]
            fp_std = fp_stats['std'].iloc[0]
            fn_std = fn_stats['std'].iloc[0]
            
            # Calculate overlap measure (simplified)
            mean_diff = abs(fp_mean - fn_mean)
            avg_std = (fp_std + fn_std) / 2
            separation_ratio = mean_diff / avg_std if avg_std > 0 else 0
            
            fp_fn_distinct[feature] = separation_ratio
    
    if fp_fn_distinct:
        print("FP vs FN separation ratios (higher = more distinct regions):")
        for feature, ratio in sorted(fp_fn_distinct.items(), key=lambda x: x[1], reverse=True):
            distinct_level = "Well-separated" if ratio > 2.0 else "Moderately separated" if ratio > 1.0 else "Overlapping"
            print(f"  {feature}: {ratio:.2f} ({distinct_level})")
        
        # Overall assessment
        avg_separation = np.mean(list(fp_fn_distinct.values()))
        if avg_separation > 2.0:
            print(f"\nOverall: FP and FN appear to occupy WELL-SEPARATED regions of directional-vote space")
        elif avg_separation > 1.0:
            print(f"\nOverall: FP and FN appear to occupy MODERATELY SEPARATED regions of directional-vote space")
        else:
            print(f"\nOverall: FP and FN appear to have OVERLAPPING regions of directional-vote space")
    else:
        print("⚠ Insufficient data to assess FP vs FN separation")
else:
    print("⚠ No summary statistics available for FP vs FN comparison")

print("\n3. Whether errors occur when the directional-vote features disagree.")
print("-" * 60)

# Check for disagreement among directional-vote features for misclassified samples
if not misclassified_examples.empty and len(available_directional_features) >= 2:
    # For each misclassified sample, check if directional-vote features show disagreement
    # We'll look at the variance/std of normalized feature values
    
    # Extract directional-vote feature values for misclassified samples
    misclassified_feature_cols = [f'{f}_value' for f in available_directional_features if f'{f}_value' in misclassified_examples.columns]
    
    if misclassified_feature_cols:
        # Normalize each feature to [0,1] scale for comparison
        feature_values_norm = misclassified_examples[misclassified_feature_cols].copy()
        for col in misclassified_feature_cols:
            min_val = feature_values_norm[col].min()
            max_val = feature_values_norm[col].max()
            if max_val > min_val:
                feature_values_norm[col] = (feature_values_norm[col] - min_val) / (max_val - min_val)
            else:
                feature_values_norm[col] = 0.5  # All same value
        
        # Calculate standard deviation across features for each sample
        misclassified_examples['feature_disagreement'] = feature_values_norm.std(axis=1)
        
        # Compare disagreement between misclassified and correctly classified samples
        # Get correctly classified samples
        correct_mask = (y == predictions)
        correct_indices = np.where(correct_mask)[0]
        
        if len(correct_indices) > 0:
            correct_feature_cols = [f'{f}_value' for f in available_directional_features if f'{f}_value' in locals().get('misclassified_examples', pd.DataFrame()).columns]
            # We need to get feature values for correct samples
            if trainer is not None:
                X_features_all = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
                feature_names_list = list(trainer.feature_names) if hasattr(trainer, 'feature_names') else []
            else:
                X_features_all = feature_engineer.transform(X, training=True).values
                feature_names_list = list(feature_engineer.get_feature_names_out()) if hasattr(feature_engineer, 'get_feature_names_out') else []
            
            directional_feature_indices = [feature_names_list.index(f) for f in available_directional_features if f in feature_names_list]
            correct_feature_values = X_features_all[correct_indices][:, directional_feature_indices]
            
            # Normalize correct features
            correct_feature_values_norm = correct_feature_values.copy()
            for i in range(len(available_directional_features)):
                col_min = correct_feature_values_norm[:, i].min()
                col_max = correct_feature_values_norm[:, i].max()
                if col_max > col_min:
                    correct_feature_values_norm[:, i] = (correct_feature_values_norm[:, i] - col_min) / (col_max - col_min)
                else:
                    correct_feature_values_norm[:, i] = 0.5
            
            correct_disagreement = np.std(correct_feature_values_norm, axis=1)
            
            # Compare mean disagreement
            misclassified_mean_disagree = np.mean(misclassified_examples['feature_disagreement'])
            correct_mean_disagree = np.mean(correct_disagreement)
            
            print(f"Mean feature disagreement (std across normalized features):")
            print(f"  Misclassified samples: {misclassified_mean_disagree:.3f}")
            print(f"  Correctly classified samples: {correct_mean_disagree:.3f}")
            
            if misclassified_mean_disagree > correct_mean_disagree * 1.2:
                print(f"  → Errors show HIGHER feature disagreement than correct predictions")
                print(f"  → Errors tend to occur when directional-vote features disagree")
            elif misclassified_mean_disagree < correct_mean_disagree * 0.8:
                print(f"  → Errors show LOWER feature disagreement than correct predictions")
                print(f"  → Errors tend to occur when directional-vote features agree")
            else:
                print(f"  → Errors show SIMILAR feature disagreement to correct predictions")
                print(f"  → No clear relationship between feature disagreement and errors")
        else:
            print("⚠ Could not compare disagreement with correct samples")
    else:
        print("⚠ Could not extract feature values for disagreement analysis")
else:
    print("⚠ Insufficient data to assess feature disagreement in errors")

print("\n4. Whether errors are predominantly high-confidence or near the classification boundary.")
print("-" * 60)

if not confidence_stats.empty:
    # Analyze FP and FN confidence levels
    fp_conf = confidence_stats[confidence_stats['group'] == 'FP'] if 'group' in confidence_stats.columns else pd.DataFrame()
    fn_conf = confidence_stats[confidence_stats['group'] == 'FN'] if 'group' in confidence_stats.columns else pd.DataFrame()
    
    confidence_assessments = []
    
    if not fp_conf.empty:
        mean_fp_prob = fp_conf['mean_probability'].iloc[0]
        if mean_fp_prob > 0.7:
            fp_assessment = "HIGH-CONFIDENCE errors"
        elif mean_fp_prob < 0.3:
            fp_assessment = "LOW-CONFIDENCE errors (near boundary)"
        else:
            fp_assessment = "MEDIUM-CONFIDENCE errors"
        confidence_assessments.append(f"False Positives: {fp_assessment} (mean prob = {mean_fp_prob:.3f})")
    
    if not fn_conf.empty:
        mean_fn_prob = fn_conf['mean_probability'].iloc[0]
        # For FN, recall that these are true class 1 predicted as 0
        # So low probability means high confidence in wrong prediction
        inv_mean_fn_prob = 1 - mean_fn_prob
        if inv_mean_fn_prob > 0.7:
            fn_assessment = "HIGH-CONFIDENCE errors"
        elif inv_mean_fn_prob < 0.3:
            fn_assessment = "LOW-CONFIDENCE errors (near boundary)"
        else:
            fn_assessment = "MEDIUM-CONFIDENCE errors"
        confidence_assessments.append(f"False Negatives: {fn_assessment} (mean prob = {mean_fn_prob:.3f})")
    
    for assessment in confidence_assessments:
        print(f"  {assessment}")
    
    # Overall assessment
    if not fp_conf.empty and not fn_conf.empty:
        fp_high_conf = fp_conf['mean_probability'].iloc[0] > 0.7
        fn_high_conf = (1 - fn_conf['mean_probability'].iloc[0]) > 0.7
        
        if fp_high_conf and fn_high_conf:
            print(f"\nOverall: BOTH FP and FN are predominantly HIGH-CONFIDENCE errors")
        elif not fp_high_conf and not fn_high_conf:
            print(f"\nOverall: BOTH FP and FN are predominantly LOW-CONFIDENCE errors (near boundary)")
        else:
            print(f"\nOverall: MIXED confidence levels - one error type high-confidence, other low/medium")
    elif not fp_conf.empty:
        fp_high_conf = fp_conf['mean_probability'].iloc[0] > 0.7
        if fp_high_conf:
            print(f"\nOverall: FP errors are predominantly HIGH-CONFIDENCE")
        else:
            print(f"\nOverall: FP errors are predominantly LOW-CONFIDENCE (near boundary)")
    elif not fn_conf.empty:
        fn_high_conf = (1 - fn_conf['mean_probability'].iloc[0]) > 0.7
        if fn_high_conf:
            print(f"\nOverall: FN errors are predominantly HIGH-CONFIDENCE")
        else:
            print(f"\nOverall: FN errors are predominantly LOW-CONFIDENCE (near boundary)")
else:
    print("⚠ No confidence statistics available")

print("\n5. Any obvious thresholds or regions where the directional vote appears unreliable.")
print("-" * 60)

# Look for regions where error rate is high
if not misclassified_examples.empty and len(available_directional_features) > 0:
    print("Regions of high error rate for each directional-vote feature:")
    
    unreliable_regions = []
    
    for feature in available_directional_features:
        feature_col = f'{feature}_value'
        if feature_col in misclassified_examples.columns:
            # Get feature values and error status
            feat_vals = misclassified_examples[feature_col].values
            true_vals = misclassified_examples['true_class'].values
            pred_vals = misclassified_examples['predicted_class'].values
            
            # Calculate error rate in bins
            if len(feat_vals) > 10:  # Only if we have enough samples
                # Create bins
                n_bins = min(5, len(np.unique(feat_vals)) // 2)
                n_bins = max(2, n_bins)
                
                if len(np.unique(feat_vals)) > n_bins:
                    bins = np.percentile(feat_vals, np.linspace(0, 100, n_bins + 1))
                    bins = np.unique(bins)
                    
                    if len(bins) > 1:
                        bin_indices = np.digitize(feat_vals, bins[:-1])
                        bin_indices = np.clip(bin_indices, 0, len(bins)-2)
                        
                        # Calculate error rate per bin
                        high_error_bins = []
                        for bin_idx in range(len(bins)-1):
                            bin_mask = (bin_indices == bin_idx)
                            if np.sum(bin_mask) > 0:
                                # Calculate error rate in this bin
                                bin_errors = np.sum((true_vals[bin_mask] != pred_vals[bin_mask]))
                                bin_total = np.sum(bin_mask)
                                bin_error_rate = bin_errors / bin_total if bin_total > 0 else 0
                                
                                if bin_error_rate > 0.5:  # More than 50% error rate
                                    bin_center = (bins[bin_idx] + bins[bin_idx+1]) / 2
                                    high_error_bins.append((bin_center, bin_error_rate, bin_total))
                        
                        if high_error_bins:
                            unreliable_regions.append((feature, high_error_bins))
    
    if unreliable_regions:
        for feature, regions in unreliable_regions:
            print(f"\n{feature}:")
            for center, error_rate, count in regions:
                print(f"  Value ~{center:.3f}: {error_rate*100:.1f}% error rate (n={count})")
    else:
        print("  No obvious high-error regions identified (may need more data or different binning)")
        
    # Also check overall error rate
    total_samples = len(y)
    total_errors = np.sum(y != predictions)
    overall_error_rate = total_errors / total_samples if total_samples > 0 else 0
    print(f"\nOverall error rate: {overall_error_rate*100:.1f}% ({total_errors}/{total_samples})")
    
    if overall_error_rate > 0.3:
        print("  → High overall error rate suggests directional vote may be unreliable broadly")
    elif overall_error_rate > 0.1:
        print("  → Moderate overall error rate suggests some unreliability in directional vote")
    else:
        print("  → Low overall error rate suggests directional vote is generally reliable")
else:
    print("⚠ Insufficient data to identify unreliable regions")

print("\n" + "="*80)
print("SUMMARY COMPLETE")
print("="*80)

# Save summary to a text file
summary_text = f"""
DIRECTIONAL VOTE ERROR ANALYSIS SUMMARY
Generated from analysis of directional-vote features in aquaculture model
Timestamp: {pd.Timestamp.now()}

Key findings would be displayed above when this cell is executed.
"""

summary_file = output_dir / 'directional_vote_error_analysis_summary.txt'
with open(summary_file, 'w') as f:
    f.write(summary_text)

print(f"\nSummary template saved to: {summary_file}")